In [ ]:
//Aşama 0’da, çalışmanın tamamının sağlam, sistematik ve tekrar üretilebilir bir deney altyapısı üzerine kurulmasını sağladık. Bu aşamada henüz veri analizi ya da model eğitimi yapmadık; bunun yerine deneyin teknik ve metodolojik temelini oluşturduk. Öncelikle tüm deney parametrelerini tek bir yapı altında toplamak için bir konfigürasyon (CONFIG) tanımladık. Bu yapı içerisinde rastgelelik sabiti (seed), train/test oranı, model listesi, çıktı klasörü ve temel kolon adları gibi kritik ayarlar yer aldı. Böylece deney boyunca kullanılan tüm parametreler merkezi ve kontrol edilebilir hale getirildi. Bu yaklaşım, ileride yapılacak değişikliklerin tüm sürece otomatik olarak yansımasını sağlarken, deney tasarımının şeffaf ve düzenli olmasına da katkı sağlar.

//Ardından tekrar üretilebilirliği (reproducibility) garanti altına almak için rastgelelik kontrolü sağlandı. Python’un random modülü, NumPy ve sistem hash ayarları sabitlenerek aynı kodun her çalıştırıldığında aynı sonuçları üretmesi sağlandı. Bu adım özellikle makine öğrenmesi çalışmalarında büyük önem taşır; çünkü sabitlenmemiş rastgelelik, train/test ayrımında veya model eğitiminde küçük ama anlamlı farklılıklara yol açabilir. Bilimsel bir çalışmada sonuçların tutarlı olması ve gerektiğinde yeniden üretilebilmesi kritik bir gerekliliktir.

//Ayrıca tüm çıktıları düzenli bir şekilde saklamak için standart bir klasör yapısı oluşturuldu. Her model için ayrı alt klasörler ve ortak (shared) çıktılar için merkezi bir alan tanımlandı. Böylece ilerleyen aşamalarda üretilecek yüzlerce grafik, metrik dosyası ve XAI çıktısı düzenli bir biçimde saklanabilecek hale getirildi. Bu yapı, hem model bazlı karşılaştırmaları kolaylaştırır hem de raporlama sürecini sistematik hale getirir. Ek olarak çalışmanın koşullarını belgelemek amacıyla bir run_info.json dosyası oluşturularak deney zamanı, kullanılan seed değeri, test oranı ve model listesi gibi bilgiler kaydedildi. Bu dosya, çalışmanın hangi ayarlar altında gerçekleştirildiğini belgelemesi açısından önemlidir.

//Son olarak, çalışmada kullanılacak model seti baştan tanımlanarak deney kapsamı netleştirildi. Bu sayede ilerleyen aşamalarda tüm modeller aynı altyapı ve aynı veri bölünmesi altında değerlendirilecektir. Özetle Aşama 0’da veri üzerinde doğrudan bir analiz yapılmamış olsa da, deneyin bilimsel bütünlüğünü ve sistematikliğini garanti altına alan temel yapı kurulmuştur. Bu aşama, ilerleyen modelleme ve açıklanabilirlik (XAI) çalışmalarının güvenilir, karşılaştırılabilir ve raporlanabilir olmasını sağlayan kritik bir başlangıçtır.

In [1]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime

# =========================
# CONFIG (tek kaynak)
# =========================
CONFIG = {
    "seed": 42,
    "test_size": 0.20,
    "top_k_features": 20,
    "label_col": "label",
    "url_col": "url",
    "output_dir": "outputs",
    # Model isimleri (senin 7 modelin)
    "models": ["LogReg", "DecisionTree", "KNN", "RandomForest", "XGBoost", "LightGBM", "CatBoost"]
}

# =========================
# Reproducibility
# =========================
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(CONFIG["seed"])

# =========================
# Output folder structure
# =========================
BASE_OUT = Path(CONFIG["output_dir"])
SHARED = BASE_OUT / "_shared"

SUBFOLDERS = ["metrics", "global", "dependence", "local_shap", "local_lime", "cases"]

def ensure_dirs():
    SHARED.mkdir(parents=True, exist_ok=True)
    for m in CONFIG["models"]:
        mdir = BASE_OUT / m
        for sf in SUBFOLDERS:
            (mdir / sf).mkdir(parents=True, exist_ok=True)

ensure_dirs()

# =========================
# Run info (log)
# =========================
run_info = {
    "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "seed": CONFIG["seed"],
    "test_size": CONFIG["test_size"],
    "top_k_features": CONFIG["top_k_features"],
    "models": CONFIG["models"],
}

with open(SHARED / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2, ensure_ascii=False)

print("Aşama 0 tamamlandı ✅")
print(f"Output dizini: {BASE_OUT.resolve()}")
print(f"Run info: {(SHARED / 'run_info.json').resolve()}")


Aşama 0 tamamlandı ✅
Output dizini: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs
Run info: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\_shared\run_info.json


In [ ]:
//Bu aşamada veri setini sisteme dahil ettik ve çalışmaya uygunluğunu doğruladık. Öncelikle CSV dosyası yüklendi, url ve label kolonlarının varlığı kontrol edildi ve modelde kullanılacak 75 feature netleştirildi. Toplam veri boyutu (579,920 satır) ve sınıf dağılımı incelendi; veri yaklaşık %58 benign ve %42 phishing olacak şekilde dengeli bulundu. Eksik değer, sonsuz değer ve tekrar eden kayıt kontrolleri yapıldı ve veri temiz olduğu doğrulandı. Ardından deney boyunca tutarlılığı sağlamak için stratified train/test split (%80–%20) gerçekleştirildi ve bu bölünme dosyaya kaydedildi. Böylece tüm modeller aynı veri bölünmesi üzerinde, adil ve tekrar üretilebilir şekilde eğitilmeye hazır hale getirildi.

In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# ---- AŞAMA 1 INPUT ----
DATA_PATH = "~/Bitirme_Projesi_Dataset_Olusturma/Makale/final_lexical_dns_whois_cleaned.csv"  # gerekirse değiştir

label_col = CONFIG["label_col"]
url_col = CONFIG["url_col"]
seed = CONFIG["seed"]
test_size = CONFIG["test_size"]

BASE_OUT = Path(CONFIG["output_dir"])
SHARED = BASE_OUT / "_shared"

# -----------------------
# 1) Load
# -----------------------
df = pd.read_csv(DATA_PATH)

# Basic column checks
assert label_col in df.columns, f"label kolonu yok: {label_col}"
assert url_col in df.columns, f"url kolonu yok: {url_col}"

# -----------------------
# 2) Build feature columns (75 features)
# -----------------------
# Varsayım: label + url dışındaki her şey feature.
feature_cols = [c for c in df.columns if c not in [label_col, url_col]]

X = df[feature_cols].copy()
y = df[label_col].copy()
urls = df[url_col].copy()

# -----------------------
# 3) Dataset profile + sanity checks
# -----------------------
profile = {
    "data_path": DATA_PATH,
    "n_rows": int(df.shape[0]),
    "n_cols_total": int(df.shape[1]),
    "n_features": int(len(feature_cols)),
    "label_col": label_col,
    "url_col": url_col,
    "label_counts": y.value_counts(dropna=False).to_dict(),
    "label_ratio": (y.value_counts(normalize=True, dropna=False).round(4)).to_dict(),
}

# Missing values per feature
missing = X.isna().sum().sort_values(ascending=False)
missing_df = missing.reset_index()
missing_df.columns = ["feature", "missing_count"]
missing_df["missing_ratio"] = (missing_df["missing_count"] / len(X)).round(6)

# Duplicate rows check (full row)
dup_count = int(df.duplicated().sum())
dup_url_label_count = int(df[[url_col, label_col]].duplicated().sum())

dup_report = {
    "duplicated_full_rows": dup_count,
    "duplicated_url_label_pairs": dup_url_label_count
}

# Inf check (numeric only)
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
inf_count = int(np.isinf(X[numeric_cols].to_numpy()).sum()) if numeric_cols else 0
profile["inf_count_numeric"] = inf_count

# Column name duplicates (rare but possible if load changed names)
# Pandas usually handles duplicates by .1 suffix; we still log suspicious near-duplicates:
suspicious_dupes = [c for c in feature_cols if c.endswith(".1")]
profile["suspicious_duplicate_columns_suffix_1"] = suspicious_dupes

# Save shared artifacts
(SHARED / "dataset_profile.json").write_text(json.dumps(profile, indent=2, ensure_ascii=False), encoding="utf-8")
missing_df.to_csv(SHARED / "missing_values.csv", index=False, encoding="utf-8")

# Save feature list
(SHARED / "feature_list_75.txt").write_text("\n".join(feature_cols), encoding="utf-8")
(SHARED / "duplicate_report.json").write_text(json.dumps(dup_report, indent=2, ensure_ascii=False), encoding="utf-8")

print("Dataset yüklendi ✅")
print(f"Satır: {df.shape[0]} | Toplam kolon: {df.shape[1]} | Feature sayısı: {len(feature_cols)}")
print("Label dağılımı:", profile["label_counts"])
print("En çok missing olan ilk 10 feature:")
display(missing_df.head(10))

# -----------------------
# 4) Stratified train/test split (standard)
# -----------------------
idx = np.arange(len(df))
train_idx, test_idx = train_test_split(
    idx,
    test_size=test_size,
    random_state=seed,
    stratify=y
)

pd.DataFrame({"train_idx": train_idx}).to_csv(SHARED / "split_indices_train.csv", index=False, encoding="utf-8")
pd.DataFrame({"test_idx": test_idx}).to_csv(SHARED / "split_indices_test.csv", index=False, encoding="utf-8")

print("Stratified split tamam ✅")
print(f"Train size: {len(train_idx)} | Test size: {len(test_idx)}")
print(f"Kayıtlar: {(SHARED / 'dataset_profile.json').resolve()}")


Dataset yüklendi ✅
Satır: 579920 | Toplam kolon: 77 | Feature sayısı: 75
Label dağılımı: {0: 339074, 1: 240846}
En çok missing olan ilk 10 feature:


,feature,missing_count,missing_ratio
0,url_length,0,0.0
1,success,0,0.0
2,query_key_count,0,0.0
3,contains_brand,0,0.0
4,contains_cloud,0,0.0
5,contains_bank,0,0.0
6,contains_update,0,0.0
7,contains_account,0,0.0
8,contains_verify,0,0.0
9,contains_secure,0,0.0


Stratified split tamam ✅
Train size: 463936 | Test size: 115984
Kayıtlar: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\_shared\dataset_profile.json


In [4]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ------------------------
# Load split indices
# ------------------------
train_idx = pd.read_csv(SHARED / "split_indices_train.csv")["train_idx"].values
test_idx  = pd.read_csv(SHARED / "split_indices_test.csv")["test_idx"].values

X_train = X.iloc[train_idx].copy()
X_test  = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test  = y.iloc[test_idx].copy()

print("Train/Test hazır ✅")
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# ------------------------
# Define preprocessors
# ------------------------

# Tree models: impute only
tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Linear / distance models: impute + scale
linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Pipeline'lar oluşturuldu ✅")


Train/Test hazır ✅
Train shape: (463936, 75)
Test shape: (115984, 75)
Pipeline'lar oluşturuldu ✅


In [ ]:
//Bu aşamada veri setindeki 75 feature’ın tamamı için sınıflar arası dağılım analizleri gerçekleştirildi. Amaç, model kurmadan önce hangi değişkenlerin phishing ve benign sınıflarını doğal olarak ayırabildiğini anlamaktı. Öncelikle sınıf dağılımı görselleştirildi, ardından her bir feature için sınıf bazlı histogram ve boxplot grafikler üretildi. Bu grafikler, feature değerlerinin iki sınıf arasında nasıl farklılaştığını görsel olarak ortaya koydu.

//Görsel analizlerin yanında, her feature için Mann–Whitney U testi uygulanarak sınıflar arası farkın istatistiksel anlamlılığı ölçüldü. Ayrıca etki büyüklüğünü değerlendirmek için rank-biserial korelasyon hesaplandı ve çoklu test düzeltmesi (FDR) uygulandı. Sonuç olarak, tüm feature’ların ayırt edicilik düzeyi sayısal olarak raporlandı ve hangi değişkenlerin daha güçlü ayrım sağladığı belirlendi.

//Bu aşama, ileride yapılacak modelleme ve XAI analizlerinin temelini oluşturdu; çünkü modelin önemli bulacağı feature’ların veri dağılımı açısından gerçekten ayırt edici olup olmadığı bu noktada ortaya konmuş oldu.

In [5]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu

# ----------------------------
# Output dirs
# ----------------------------
DIST_DIR = SHARED / "feature_distributions"
HIST_DIR = DIST_DIR / "hist"
BOX_DIR  = DIST_DIR / "box"
HIST_DIR.mkdir(parents=True, exist_ok=True)
BOX_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 2.1 Class distribution plot
# ----------------------------
counts = y.value_counts().sort_index()
ratios = y.value_counts(normalize=True).sort_index()

plt.figure()
plt.bar(counts.index.astype(str), counts.values)
plt.title("Class Distribution (0=Benign, 1=Phishing)")
plt.xlabel("Class")
plt.ylabel("Count")
for i, (c, r) in enumerate(zip(counts.values, ratios.values)):
    plt.text(i, c, f"{c}\n({r:.2%})", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(SHARED / "class_distribution.png", dpi=200)
plt.close()

print("✅ class_distribution.png kaydedildi")

# ----------------------------
# Helpers
# ----------------------------
def safe_series(s: pd.Series):
    """Inf ve NaN temizliği (sizde NaN yok ama güvenlik için)."""
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    return s

def rank_biserial_from_u(u_stat, n1, n2):
    # rank-biserial correlation = 1 - (2U)/(n1*n2)  (class1 vs class0 yönüne göre işaret değişebilir)
    # burada class1'i 'x', class0'ı 'y' aldığımız için:
    return 1 - (2*u_stat)/(n1*n2)

# ----------------------------
# 2.2 + 2.3 Feature plots + stats
# ----------------------------
stats_rows = []

y0_mask = (y == 0)
y1_mask = (y == 1)

for feat in feature_cols:
    s0 = safe_series(df.loc[y0_mask, feat])
    s1 = safe_series(df.loc[y1_mask, feat])

    # ----- stats -----
    med0 = float(np.median(s0)) if len(s0) else np.nan
    med1 = float(np.median(s1)) if len(s1) else np.nan

    # Mann–Whitney U (two-sided)
    # large sample -> scipy ok
    try:
        u_stat, p_val = mannwhitneyu(s1, s0, alternative="two-sided")
        rbc = rank_biserial_from_u(u_stat, len(s1), len(s0))
    except Exception:
        p_val = np.nan
        rbc = np.nan

    stats_rows.append({
        "feature": feat,
        "median_class0": med0,
        "median_class1": med1,
        "p_value": p_val,
        "rank_biserial": rbc
    })

    # ----- histogram plot -----
    plt.figure()
    plt.hist(s0, bins=50, density=True, alpha=0.6, label="Benign (0)")
    plt.hist(s1, bins=50, density=True, alpha=0.6, label="Phishing (1)")
    plt.title(f"Histogram: {feat}")
    plt.xlabel(feat)
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(HIST_DIR / f"{feat}.png", dpi=200)
    plt.close()

    # ----- boxplot -----
    plt.figure()
    plt.boxplot([s0.values, s1.values], labels=["Benign (0)", "Phishing (1)"], showfliers=False)
    plt.title(f"Boxplot: {feat}")
    plt.ylabel(feat)
    plt.tight_layout()
    plt.savefig(BOX_DIR / f"{feat}.png", dpi=200)
    plt.close()

print("✅ 75 feature için histogram + boxplot kaydedildi")

# ----------------------------
# Multiple testing correction (BH-FDR)
# ----------------------------
stats_df = pd.DataFrame(stats_rows)

# BH-FDR
p = stats_df["p_value"].values
valid = np.isfinite(p)
p_valid = p[valid]

order = np.argsort(p_valid)
ranked = p_valid[order]
m = len(ranked)
q = np.empty(m)

# q_i = p_i * m / i
for i in range(m):
    q[i] = ranked[i] * m / (i+1)

# enforce monotonicity
q = np.minimum.accumulate(q[::-1])[::-1]

# map back
q_full = np.full_like(p, np.nan, dtype=float)
q_full_valid = np.empty_like(p_valid)
q_full_valid[order] = q
q_full[valid] = q_full_valid

stats_df["p_fdr_bh"] = q_full
stats_df["abs_rank_biserial"] = stats_df["rank_biserial"].abs()

stats_df.to_csv(SHARED / "feature_stats.csv", index=False, encoding="utf-8")
print("✅ feature_stats.csv kaydedildi")

# ----------------------------
# Top 20 by effect size (abs rank-biserial)
# ----------------------------
top20 = stats_df.sort_values("abs_rank_biserial", ascending=False).head(20)

plt.figure(figsize=(10, 6))
plt.barh(top20["feature"][::-1], top20["abs_rank_biserial"][::-1])
plt.title("Top 20 Features by Effect Size (|rank-biserial|)")
plt.xlabel("|rank-biserial|")
plt.tight_layout()
plt.savefig(SHARED / "top20_features_stats.png", dpi=200)
plt.close()

top20.to_csv(SHARED / "top20_feature_stats.csv", index=False, encoding="utf-8")
print("✅ top20_features_stats.png ve top20_feature_stats.csv kaydedildi")

print("Aşama 2 tamamlandı ✅")
print("Önemli dosyalar:")
print(" -", (SHARED / "class_distribution.png").resolve())
print(" -", (SHARED / "feature_stats.csv").resolve())
print(" -", (SHARED / "top20_features_stats.png").resolve())


✅ class_distribution.png kaydedildi


C:\Users\elifo\AppData\Local\Temp\ipykernel_10580\1644010372.py:97: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([s0.values, s1.values], labels=["Benign (0)", "Phishing (1)"], showfliers=False)
C:\Users\elifo\AppData\Local\Temp\ipykernel_10580\1644010372.py:97: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([s0.values, s1.values], labels=["Benign (0)", "Phishing (1)"], showfliers=False)
C:\Users\elifo\AppData\Local\Temp\ipykernel_10580\1644010372.py:97: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([s0.values, s1.values], labels=["Benign (0)", "Phishing (1)"], showfliers=False)
C:\U

✅ 75 feature için histogram + boxplot kaydedildi
✅ feature_stats.csv kaydedildi
✅ top20_features_stats.png ve top20_feature_stats.csv kaydedildi
Aşama 2 tamamlandı ✅
Önemli dosyalar:
 - C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\_shared\class_distribution.png
 - C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\_shared\feature_stats.csv
 - C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\_shared\top20_features_stats.png


In [ ]:
//Bu aşamada, oluşturulan 75 feature üzerinden belirlenen yedi farklı makine öğrenmesi modeli eğitildi ve karşılaştırıldı. Amaç, veri üzerinde hangi modelin daha iyi genelleme yaptığını ve phishing tespitinde daha yüksek başarı sağladığını ölçmekti. Eğitim sürecinde, Aşama 1’de oluşturulan %80 train / %20 test (stratified) veri bölünmesi kullanıldı. Böylece tüm modeller aynı veri koşulları altında eğitildi ve adil bir karşılaştırma ortamı sağlandı.

//Her model için uygun ön işleme adımları uygulandı: ağaç tabanlı modellerde yalnızca eksik değer tamamlama (median imputation) yapılırken, lojistik regresyon ve KNN gibi ölçeğe duyarlı modellerde standardizasyon da uygulandı. Modeller train verisi üzerinde eğitildi ve performansları yalnızca test verisi üzerinde ölçüldü; böylece veri sızıntısı (data leakage) engellendi.

//Değerlendirme metrikleri olarak accuracy, precision, recall, F1-score, ROC-AUC ve PR-AUC hesaplandı. Özellikle phishing sınıfı (pozitif sınıf) için precision, recall ve F1-score değerleri ayrı olarak raporlandı. Ayrıca her model için confusion matrix, ROC eğrisi ve Precision–Recall eğrisi oluşturularak görsel performans analizi yapıldı. Tüm sonuçlar sistematik şekilde kaydedildi ve modeller F1 veya PR-AUC gibi metriklere göre sıralandı.

//Bu aşama sonucunda, veri üzerinde en iyi performans gösteren modeller belirlendi ve bir sonraki adım olan hiperparametre optimizasyonu ve açıklanabilirlik (XAI) analizleri için temel oluşturuldu.

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Optional libs (may not exist in some envs)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

# ------------------------
# Load split indices
# ------------------------
train_idx = pd.read_csv(SHARED / "split_indices_train.csv")["train_idx"].values
test_idx  = pd.read_csv(SHARED / "split_indices_test.csv")["test_idx"].values

X_train = X.iloc[train_idx].copy()
X_test  = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test  = y.iloc[test_idx].copy()

# ------------------------
# Output helpers
# ------------------------
BASE_OUT = Path(CONFIG["output_dir"])

def save_cm(y_true, y_pred, out_path, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    plt.figure()
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks([0,1], ["0", "1"])
    plt.yticks([0,1], ["0", "1"])
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def save_roc(model, Xte, yte, out_path, title):
    plt.figure()
    RocCurveDisplay.from_estimator(model, Xte, yte)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def save_pr(model, Xte, yte, out_path, title):
    plt.figure()
    PrecisionRecallDisplay.from_estimator(model, Xte, yte)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def get_proba_or_score(model, Xte):
    # Try predict_proba -> positive class proba
    if hasattr(model, "predict_proba"):
        return model.predict_proba(Xte)[:, 1]
    # Try decision_function
    if hasattr(model, "decision_function"):
        s = model.decision_function(Xte)
        # normalize to 0-1 like score (for AUC works without but keep stable for PR-AUC)
        s = (s - s.min()) / (s.max() - s.min() + 1e-12)
        return s
    # fallback: predicted labels
    return model.predict(Xte)

# ------------------------
# Define models (baseline)
# ------------------------
models = {}

models["LogReg"] = LogisticRegression(
    max_iter=2000, n_jobs=-1, random_state=CONFIG["seed"]
)

models["DecisionTree"] = DecisionTreeClassifier(
    random_state=CONFIG["seed"]
)

models["KNN"] = KNeighborsClassifier(
    n_neighbors=5
)

models["RandomForest"] = RandomForestClassifier(
    n_estimators=300,
    random_state=CONFIG["seed"],
    n_jobs=-1
)

if HAS_XGB:
    models["XGBoost"] = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=CONFIG["seed"],
        eval_metric="logloss",
        n_jobs=-1
    )
else:
    print("⚠️ XGBoost yok -> XGBoost modeli atlanacak")

if HAS_LGBM:
    models["LightGBM"] = LGBMClassifier(
        n_estimators=800,
        learning_rate=0.05,
        num_leaves=63,
        random_state=CONFIG["seed"],
        n_jobs=-1
    )
else:
    print("⚠️ LightGBM yok -> LightGBM modeli atlanacak")

if HAS_CAT:
    models["CatBoost"] = CatBoostClassifier(
        iterations=800,
        learning_rate=0.05,
        depth=8,
        random_seed=CONFIG["seed"],
        verbose=False
    )
else:
    print("⚠️ CatBoost yok -> CatBoost modeli atlanacak")

# ------------------------
# Choose preprocessing per model type
# ------------------------
LINEAR_DISTANCE = {"LogReg", "KNN"}  # scale needed
TREE_LIKE = set(models.keys()) - LINEAR_DISTANCE

results = []

for name, clf in models.items():
    print(f"\n=== Training: {name} ===")

    # build pipeline
    if name in LINEAR_DISTANCE:
        pipe = Pipeline([
            ("preprocess", linear_preprocessor),
            ("model", clf)
        ])
    else:
        pipe = Pipeline([
            ("preprocess", tree_preprocessor),
            ("model", clf)
        ])

    # fit
    pipe.fit(X_train, y_train)

    # predict
    y_pred = pipe.predict(X_test)
    y_score = get_proba_or_score(pipe, X_test)

    # metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

    # AUCs
    try:
        roc_auc = roc_auc_score(y_test, y_score)
    except Exception:
        roc_auc = np.nan
    try:
        pr_auc = average_precision_score(y_test, y_score)
    except Exception:
        pr_auc = np.nan

    results.append({
        "model": name,
        "accuracy": acc,
        "precision_phishing": prec,
        "recall_phishing": rec,
        "f1_phishing": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    })

    # save plots
    mdir = BASE_OUT / name / "metrics"
    save_cm(y_test, y_pred, mdir / "confusion_matrix.png", f"Confusion Matrix - {name}")
    save_roc(pipe, X_test, y_test, mdir / "roc_curve.png", f"ROC Curve - {name}")
    save_pr(pipe, X_test, y_test, mdir / "pr_curve.png", f"PR Curve - {name}")

    # save per-model metrics
    pd.DataFrame([results[-1]]).to_csv(mdir / "metrics.json.csv", index=False, encoding="utf-8")
    print("✅ metrics + plots saved:", mdir)

# ------------------------
# Save summary table
# ------------------------
metrics_df = pd.DataFrame(results).sort_values("f1_phishing", ascending=False)
metrics_df.to_csv(SHARED / "metrics_table.csv", index=False, encoding="utf-8")

print("\n✅ Aşama 3 tamamlandı")
print("Özet tablo:", (SHARED / "metrics_table.csv").resolve())
display(metrics_df)



=== Training: LogReg ===
✅ metrics + plots saved: outputs\LogReg\metrics

=== Training: DecisionTree ===
✅ metrics + plots saved: outputs\DecisionTree\metrics

=== Training: KNN ===
✅ metrics + plots saved: outputs\KNN\metrics

=== Training: RandomForest ===
✅ metrics + plots saved: outputs\RandomForest\metrics

=== Training: XGBoost ===
✅ metrics + plots saved: outputs\XGBoost\metrics

=== Training: LightGBM ===
[LightGBM] [Info] Number of positive: 192677, number of negative: 271259
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.074345 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6057
[LightGBM] [Info] Number of data points in the train set: 463936, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342059
[LightGBM] [Info] Start training from score -0.342059


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ metrics + plots saved: outputs\LightGBM\metrics

=== Training: CatBoost ===
✅ metrics + plots saved: outputs\CatBoost\metrics

✅ Aşama 3 tamamlandı
Özet tablo: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\_shared\metrics_table.csv


,model,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc,pr_auc
5,LightGBM,0.986809,0.989833,0.978285,0.984025,0.998488,0.998241
3,RandomForest,0.985145,0.990496,0.973572,0.981961,0.998327,0.997946
6,CatBoost,0.983153,0.986259,0.972991,0.979580,0.997907,0.997554
4,XGBoost,0.981592,0.984385,0.971081,0.977688,0.997635,0.997215
1,DecisionTree,0.974884,0.972105,0.967282,0.969688,0.973783,0.953888
2,KNN,0.973953,0.981301,0.955490,0.968224,0.990063,0.985444
0,LogReg,0.943984,0.943867,0.919824,0.931690,0.984539,0.980366


<Figure size 640x480 with 0 Axes>

...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [ ]:
//Bu aşamada, eğitilmiş modellerin karar mekanizmasını açıklanabilir hale getirmek amacıyla global düzeyde XAI analizleri gerçekleştirdik. Model performansını yalnızca metriklerle değerlendirmek yerine, her modelin hangi feature’lara dayanarak karar verdiğini sistematik olarak ortaya koyduk. Bu analizler test seti üzerinde yapıldı; böylece modelin yalnızca eğitim verisindeki davranışı değil, görmediği veri üzerindeki genelleme davranışı açıklanmış oldu. Bu yaklaşım metodolojik olarak daha güvenilirdir ve gerçek dünya kullanım senaryosunu daha doğru temsil eder.

//Öncelikle tüm modeller için Permutation Importance hesaplandı. Bu yöntem modelden bağımsızdır ve her bir feature’ın değerleri rastgele karıştırıldığında model performansındaki düşüşü ölçer. Böylece hangi feature’ın model için kritik olduğu, doğrudan performans kaybı üzerinden sayısal olarak belirlendi. Ardından her model için SHAP (SHapley Additive Explanations) analizi uygulandı. Ağaç tabanlı modellerde TreeExplainer, lojistik regresyonda LinearExplainer ve KNN’de örneklemeli KernelExplainer kullanıldı. SHAP sayesinde her feature’ın model çıktısına yaptığı ortalama mutlak katkı hesaplandı ve global önem sıralaması oluşturuldu.

//Bu aşamanın sonunda, her model için 75 feature’ın tamamını kapsayan önem tabloları ve görsel grafikler üretildi. Böylece yalnızca “hangi model daha iyi?” sorusu değil, aynı zamanda “hangi model hangi feature’lara dayanıyor?” sorusu da yanıtlanmış oldu. Bu analiz, ilerleyen aşamalarda yapılacak model karşılaştırması, feature stabilitesi incelemesi ve detaylı yerel (local) açıklamalar için güçlü bir temel oluşturdu.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Model objelerini (Aşama 3'teki baseline ayarlar)
base_models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=CONFIG["seed"]),
    "DecisionTree": DecisionTreeClassifier(random_state=CONFIG["seed"]),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=CONFIG["seed"], n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=CONFIG["seed"], eval_metric="logloss", n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=800, learning_rate=0.05, num_leaves=63,
        random_state=CONFIG["seed"], n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=800, learning_rate=0.05, depth=8,
        random_seed=CONFIG["seed"], verbose=False
    )
}

LINEAR_DISTANCE = {"LogReg", "KNN"}  # scale gerekir
TREE_MODELS = {"DecisionTree", "RandomForest", "XGBoost", "LightGBM", "CatBoost"}

pipelines = {}

for name, clf in base_models.items():
    print(f"Fitting pipeline: {name}")
    pre = linear_preprocessor if name in LINEAR_DISTANCE else tree_preprocessor
    pipe = Pipeline([("preprocess", pre), ("model", clf)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

print("✅ Tüm pipeline'lar hazır")


Fitting pipeline: LogReg
Fitting pipeline: DecisionTree
Fitting pipeline: KNN
Fitting pipeline: RandomForest
Fitting pipeline: XGBoost
Fitting pipeline: LightGBM
[LightGBM] [Info] Number of positive: 192677, number of negative: 271259
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053680 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6057
[LightGBM] [Info] Number of data points in the train set: 463936, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342059
[LightGBM] [Info] Start training from score -0.342059
Fitting pipeline: CatBoost
✅ Tüm pipeline'lar hazır


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

perm_out = SHARED / "permutation_importance_all_models"
perm_out.mkdir(parents=True, exist_ok=True)

perm_summary = []

for name, pipe in pipelines.items():
    print(f"\nPermutation importance: {name}")
    r = permutation_importance(
        pipe, X_test, y_test,
        n_repeats=3,              # büyük test set -> 3 yeterli
        random_state=CONFIG["seed"],
        scoring="f1",
        n_jobs=-1
    )

    df_perm = pd.DataFrame({
        "feature": feature_cols,
        "importance_mean": r.importances_mean,
        "importance_std": r.importances_std
    }).sort_values("importance_mean", ascending=False)

    out_dir = BASE_OUT / name / "global"
    out_dir.mkdir(parents=True, exist_ok=True)

    df_perm.to_csv(out_dir / "perm_importance.csv", index=False, encoding="utf-8")

    # Top20 plot
    top = df_perm.head(20)
    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"][::-1], top["importance_mean"][::-1])
    plt.title(f"{name} - Permutation Importance (Top 20) [scoring=f1]")
    plt.xlabel("Mean Importance (ΔF1)")
    plt.tight_layout()
    plt.savefig(out_dir / "perm_bar_top20.png", dpi=200)
    plt.close()

    perm_summary.append({
        "model": name,
        "top1_feature": df_perm.iloc[0]["feature"],
        "top1_importance": float(df_perm.iloc[0]["importance_mean"])
    })

pd.DataFrame(perm_summary).to_csv(perm_out / "perm_summary_top1.csv", index=False, encoding="utf-8")
print("\n✅ Permutation importance tamamlandı (tüm modeller)")



Permutation importance: LogReg

Permutation importance: DecisionTree

Permutation importance: KNN

Permutation importance: RandomForest

Permutation importance: XGBoost

Permutation importance: LightGBM


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Permutation importance: CatBoost

✅ Permutation importance tamamlandı (tüm modeller)


In [4]:
#  kernel restart yaptım geri yükleme içim:
import os, random
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# -------- CONFIG --------
CONFIG = {
    "seed": 42,
    "test_size": 0.20,
    "label_col": "label",
    "url_col": "url",
    "output_dir": "outputs",
}

DATA_PATH = "~/Bitirme_Projesi_Dataset_Olusturma/Makale/final_lexical_dns_whois_cleaned.csv"

# -------- Seeds --------
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(CONFIG["seed"])

# -------- Paths --------
BASE_OUT = Path(CONFIG["output_dir"])
SHARED = BASE_OUT / "_shared"

# -------- Load dataset --------
df = pd.read_csv(DATA_PATH)
label_col = CONFIG["label_col"]
url_col = CONFIG["url_col"]

feature_cols = [c for c in df.columns if c not in [label_col, url_col]]
X = df[feature_cols].copy()
y = df[label_col].copy()

# -------- Load split indices --------
train_idx = pd.read_csv(SHARED / "split_indices_train.csv")["train_idx"].values
test_idx  = pd.read_csv(SHARED / "split_indices_test.csv")["test_idx"].values

X_train = X.iloc[train_idx].copy()
X_test  = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test  = y.iloc[test_idx].copy()

# -------- Preprocessors --------
tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# -------- Models (baseline Aşama 3 ile aynı) --------
base_models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=CONFIG["seed"]),
    "DecisionTree": DecisionTreeClassifier(random_state=CONFIG["seed"]),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=CONFIG["seed"], n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=CONFIG["seed"], eval_metric="logloss", n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=800, learning_rate=0.05, num_leaves=63,
        random_state=CONFIG["seed"], n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=800, learning_rate=0.05, depth=8,
        random_seed=CONFIG["seed"], verbose=False
    )
}

LINEAR_DISTANCE = {"LogReg", "KNN"}

# -------- Fit pipelines --------
pipelines = {}
for name, clf in base_models.items():
    print("Fitting:", name)
    pre = linear_preprocessor if name in LINEAR_DISTANCE else tree_preprocessor
    pipe = Pipeline([("preprocess", pre), ("model", clf)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

print("\n✅ Kernel reset sonrası SHAP için her şey hazır")
print("X_train:", X_train.shape, "X_test:", X_test.shape, "features:", len(feature_cols))


Fitting: LogReg
Fitting: DecisionTree
Fitting: KNN
Fitting: RandomForest
Fitting: XGBoost
Fitting: LightGBM
[LightGBM] [Info] Number of positive: 192677, number of negative: 271259
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.063712 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6057
[LightGBM] [Info] Number of data points in the train set: 463936, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342059
[LightGBM] [Info] Start training from score -0.342059
Fitting: CatBoost

✅ Kernel reset sonrası SHAP için her şey hazır
X_train: (463936, 75) X_test: (115984, 75) features: 75


In [6]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PLOT_SAMPLE = 20000
KNN_SHAP_BG = 2000
KNN_SHAP_TEST = 5000

rng = np.random.RandomState(CONFIG["seed"])

def to_2d_shap(shap_vals):
    """SHAP çıktısını her zaman (n_samples, n_features) şekline getirir."""
    if isinstance(shap_vals, list):
        # binary: [class0, class1]
        if len(shap_vals) >= 2:
            return np.asarray(shap_vals[1])
        return np.asarray(shap_vals[-1])

    arr = np.asarray(shap_vals)

    if arr.ndim == 2:
        return arr

    if arr.ndim == 3:
        # (n_samples, n_features, n_classes) ise class=1 al
        if arr.shape[-1] >= 2:
            return arr[:, :, 1]
        return np.squeeze(arr, axis=-1)

    return np.squeeze(arr)

def save_shap_bar(df_shap, out_path, title, topn=20):
    top = df_shap.head(topn)
    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"][::-1], top["mean_abs_shap"][::-1])
    plt.title(title)
    plt.xlabel("mean(|SHAP|)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

TREE_MODELS = {"DecisionTree", "RandomForest", "XGBoost", "LightGBM", "CatBoost"}

for name, pipe in pipelines.items():
    print(f"\nSHAP global: {name}")

    out_dir = BASE_OUT / name / "global"
    out_dir.mkdir(parents=True, exist_ok=True)

    Xte_proc = pipe.named_steps["preprocess"].transform(X_test)
    Xtr_proc = pipe.named_steps["preprocess"].transform(X_train)
    model = pipe.named_steps["model"]

    # 1) Tree models
    if name in TREE_MODELS:
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(Xte_proc,check_additivity=False)
        shap_mat = to_2d_shap(shap_vals)

        mean_abs = np.abs(shap_mat).mean(axis=0)  # <-- artık 1D
        df_shap = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs}) \
                    .sort_values("mean_abs_shap", ascending=False)

        df_shap.to_csv(out_dir / "shap_importance.csv", index=False, encoding="utf-8")
        save_shap_bar(df_shap, out_dir / "shap_bar_top20.png", f"{name} - SHAP Importance (Top 20)")

        # summary plot sample
        n = Xte_proc.shape[0]
        idx_plot = rng.choice(n, size=min(PLOT_SAMPLE, n), replace=False)
        plt.figure()
        shap.summary_plot(shap_mat[idx_plot], Xte_proc[idx_plot], feature_names=feature_cols, show=False)
        plt.tight_layout()
        plt.savefig(out_dir / "shap_summary.png", dpi=200)
        plt.close()

        print("✅ Tree SHAP tamam")

    # 2) Logistic Regression
    elif name == "LogReg":
        ntr = Xtr_proc.shape[0]
        idx_bg = rng.choice(ntr, size=min(20000, ntr), replace=False)
        bg = Xtr_proc[idx_bg]

        explainer = shap.LinearExplainer(model, bg)
        shap_mat = explainer.shap_values(Xte_proc)
        shap_mat = to_2d_shap(shap_mat)

        mean_abs = np.abs(shap_mat).mean(axis=0)
        df_shap = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs}) \
                    .sort_values("mean_abs_shap", ascending=False)

        df_shap.to_csv(out_dir / "shap_importance.csv", index=False, encoding="utf-8")
        save_shap_bar(df_shap, out_dir / "shap_bar_top20.png", f"{name} - SHAP Importance (Top 20)")

        n = Xte_proc.shape[0]
        idx_plot = rng.choice(n, size=min(PLOT_SAMPLE, n), replace=False)
        plt.figure()
        shap.summary_plot(shap_mat[idx_plot], Xte_proc[idx_plot], feature_names=feature_cols, show=False)
        plt.tight_layout()
        plt.savefig(out_dir / "shap_summary.png", dpi=200)
        plt.close()

        print("✅ Linear SHAP tamam")

    # 3) KNN (sampled Kernel SHAP)
    elif name == "KNN":
        ntr = Xtr_proc.shape[0]
        nte = Xte_proc.shape[0]

        idx_bg = rng.choice(ntr, size=min(KNN_SHAP_BG, ntr), replace=False)
        idx_te = rng.choice(nte, size=min(KNN_SHAP_TEST, nte), replace=False)

        bg = Xtr_proc[idx_bg]
        X_small = Xte_proc[idx_te]

        f = lambda x: model.predict_proba(x)[:, 1]
        explainer = shap.KernelExplainer(f, bg)
        shap_mat = explainer.shap_values(X_small, nsamples=200)
        shap_mat = to_2d_shap(shap_mat)

        mean_abs = np.abs(shap_mat).mean(axis=0)
        df_shap = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs}) \
                    .sort_values("mean_abs_shap", ascending=False)

        df_shap.to_csv(out_dir / "shap_importance.csv", index=False, encoding="utf-8")
        save_shap_bar(df_shap, out_dir / "shap_bar_top20.png", f"{name} - SHAP Importance (Top 20) [sampled]")

        plt.figure()
        shap.summary_plot(shap_mat, X_small, feature_names=feature_cols, show=False)
        plt.tight_layout()
        plt.savefig(out_dir / "shap_summary.png", dpi=200)
        plt.close()

        meta = {"method":"KernelExplainer (sampled)", "background_size":int(bg.shape[0]),
                "test_explained_size":int(X_small.shape[0]), "nsamples":200}
        pd.Series(meta).to_csv(out_dir / "shap_sampling_meta.csv", encoding="utf-8")

        print("✅ KNN SHAP tamam (örneklemeli)")

print("\n✅ Hücre 3 bitti: tüm modeller için SHAP global üretildi")



SHAP global: LogReg


C:\Users\elifo\AppData\Local\Temp\ipykernel_22932\3628385731.py:99: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_mat[idx_plot], Xte_proc[idx_plot], feature_names=feature_cols, show=False)


✅ Linear SHAP tamam

SHAP global: DecisionTree


C:\Users\elifo\AppData\Local\Temp\ipykernel_22932\3628385731.py:72: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_mat[idx_plot], Xte_proc[idx_plot], feature_names=feature_cols, show=False)


✅ Tree SHAP tamam

SHAP global: KNN


Using 2000 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/5000 [00:00<?, ?it/s]


KeyboardInterrupt



In [ ]:
// desicion tree ve logreg sonrasını tekrar knn için çalıştırdım. knn ayarları yüksek ölçüde küçültüldü

In [8]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ✅ KNN için hızlandırılmış ayarlar (sadece KNN değişiyor)
KNN_SHAP_BG = 200      # 2000 -> 200  (10x hızlı)
KNN_SHAP_TEST = 100    # 1000 -> 100  (10x hızlı)
KNN_NSAMPLES = 50      # 100  -> 50   (2x hızlı)

rng = np.random.RandomState(CONFIG["seed"])

def to_2d_shap(shap_vals):
    if isinstance(shap_vals, list):
        if len(shap_vals) >= 2:
            return np.asarray(shap_vals[1])
        return np.asarray(shap_vals[-1])
    arr = np.asarray(shap_vals)
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3:
        if arr.shape[-1] >= 2:
            return arr[:, :, 1]
        return np.squeeze(arr, axis=-1)
    return np.squeeze(arr)

def save_shap_bar(df_shap, out_path, title, topn=20):
    top = df_shap.head(topn)
    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"][::-1], top["mean_abs_shap"][::-1])
    plt.title(title)
    plt.xlabel("mean(|SHAP|)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

name = "KNN"
pipe = pipelines[name]

out_dir = BASE_OUT / name / "global"
out_dir.mkdir(parents=True, exist_ok=True)

Xte_proc = pipe.named_steps["preprocess"].transform(X_test)
Xtr_proc = pipe.named_steps["preprocess"].transform(X_train)
model = pipe.named_steps["model"]

ntr = Xtr_proc.shape[0]
nte = Xte_proc.shape[0]

idx_bg = rng.choice(ntr, size=min(KNN_SHAP_BG, ntr), replace=False)
idx_te = rng.choice(nte, size=min(KNN_SHAP_TEST, nte), replace=False)

bg = Xtr_proc[idx_bg]
X_small = Xte_proc[idx_te]

f = lambda x: model.predict_proba(x)[:, 1]
explainer = shap.KernelExplainer(f, bg)

shap_vals = explainer.shap_values(X_small, nsamples=KNN_NSAMPLES)
shap_mat = to_2d_shap(shap_vals)

mean_abs = np.abs(shap_mat).mean(axis=0)
df_shap = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs}) \
            .sort_values("mean_abs_shap", ascending=False)

(df_shap).to_csv(out_dir / "shap_importance.csv", index=False, encoding="utf-8")
save_shap_bar(df_shap, out_dir / "shap_bar_top20.png", "KNN - SHAP Importance (Top 20) [sampled]")

plt.figure()
shap.summary_plot(shap_mat, X_small, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.savefig(out_dir / "shap_summary.png", dpi=200)
plt.close()

meta = {
    "method": "KernelExplainer (sampled)",
    "background_size": int(bg.shape[0]),
    "test_explained_size": int(X_small.shape[0]),
    "nsamples": int(KNN_NSAMPLES),
    "note": "Only KNN was sampled for feasibility; other models used full test set."
}
pd.Series(meta).to_csv(out_dir / "shap_meta.csv", encoding="utf-8")

print("✅ KNN SHAP tamamlandı (hızlandırılmış)")
print("Saved:", out_dir / "shap_importance.csv")


Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/100 [00:00<?, ?it/s]

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 5 iterations, i.e. alpha=2.496e-02, with an active set of 5 regressors, and the smallest cholesky pivot element being 4.215e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 5 iterations, i.e. alpha=2.167e-02, with an active set of 5 regressors, and the smallest cholesky pivot element being 4.215e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\linear_model\_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 8 iterations, i.e. alpha=1.083e-02, with an active set of 8 regressors, and t

✅ KNN SHAP tamamlandı (hızlandırılmış)
Saved: outputs\KNN\global\shap_importance.csv


In [1]:
#  kernel restart yaptım geri yükleme içim:
import os, random
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# -------- CONFIG --------
CONFIG = {
    "seed": 42,
    "test_size": 0.20,
    "label_col": "label",
    "url_col": "url",
    "output_dir": "outputs",
}

DATA_PATH = "~/Bitirme_Projesi_Dataset_Olusturma/Makale/final_lexical_dns_whois_cleaned.csv"

# -------- Seeds --------
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(CONFIG["seed"])

# -------- Paths --------
BASE_OUT = Path(CONFIG["output_dir"])
SHARED = BASE_OUT / "_shared"

# -------- Load dataset --------
df = pd.read_csv(DATA_PATH)
label_col = CONFIG["label_col"]
url_col = CONFIG["url_col"]

feature_cols = [c for c in df.columns if c not in [label_col, url_col]]
X = df[feature_cols].copy()
y = df[label_col].copy()

# -------- Load split indices --------
train_idx = pd.read_csv(SHARED / "split_indices_train.csv")["train_idx"].values
test_idx  = pd.read_csv(SHARED / "split_indices_test.csv")["test_idx"].values

X_train = X.iloc[train_idx].copy()
X_test  = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test  = y.iloc[test_idx].copy()

# -------- Preprocessors --------
tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# -------- Models (baseline Aşama 3 ile aynı) --------
base_models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=CONFIG["seed"]),
    "DecisionTree": DecisionTreeClassifier(random_state=CONFIG["seed"]),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=CONFIG["seed"], n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=CONFIG["seed"], eval_metric="logloss", n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=800, learning_rate=0.05, num_leaves=63,
        random_state=CONFIG["seed"], n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=800, learning_rate=0.05, depth=8,
        random_seed=CONFIG["seed"], verbose=False
    )
}

LINEAR_DISTANCE = {"LogReg", "KNN"}

# -------- Fit pipelines --------
pipelines = {}
for name, clf in base_models.items():
    print("Fitting:", name)
    pre = linear_preprocessor if name in LINEAR_DISTANCE else tree_preprocessor
    pipe = Pipeline([("preprocess", pre), ("model", clf)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

print("\n✅ Kernel reset sonrası SHAP için her şey hazır")
print("X_train:", X_train.shape, "X_test:", X_test.shape, "features:", len(feature_cols))


Fitting: LogReg
Fitting: DecisionTree
Fitting: KNN
Fitting: RandomForest
Fitting: XGBoost
Fitting: LightGBM
[LightGBM] [Info] Number of positive: 192677, number of negative: 271259
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.067673 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6057
[LightGBM] [Info] Number of data points in the train set: 463936, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342059
[LightGBM] [Info] Start training from score -0.342059
Fitting: CatBoost

✅ Kernel reset sonrası SHAP için her şey hazır
X_train: (463936, 75) X_test: (115984, 75) features: 75


In [4]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# =========================
# AYARLAR
# =========================
CHUNK_SIZE  = 2000    # ilerleme sıklığı (1000 daha sık)
PLOT_SAMPLE = 5000    # summary plot için örnek sayısı
SEED        = CONFIG["seed"]

MODELS = ["LogReg", "DecisionTree", "RandomForest", "XGBoost", "LightGBM", "CatBoost"]  # ✅ KNN yok
TREE_MODELS = {"DecisionTree", "RandomForest", "XGBoost", "LightGBM", "CatBoost"}

rng = np.random.RandomState(SEED)

def to_2d_shap(shap_vals):
    if isinstance(shap_vals, list):
        if len(shap_vals) >= 2:
            return np.asarray(shap_vals[1])  # class=1
        return np.asarray(shap_vals[-1])
    arr = np.asarray(shap_vals)
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3:
        if arr.shape[-1] >= 2:
            return arr[:, :, 1]
        return np.squeeze(arr, axis=-1)
    return np.squeeze(arr)

def save_shap_bar(df_shap, out_path, title, topn=20):
    top = df_shap.head(topn)
    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"][::-1], top["mean_abs_shap"][::-1])
    plt.title(title)
    plt.xlabel("mean(|SHAP|)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

print(f"✅ Full test üzerinde çalışıyoruz: X_test={X_test.shape}, seed={SEED}", flush=True)

for name in MODELS:
    t_model = time.time()
    print(f"\n============================", flush=True)
    print(f"🚀 SHAP FULL TEST: {name}", flush=True)
    print(f"============================", flush=True)

    pipe = pipelines[name]
    out_dir = BASE_OUT / name / "global"
    out_dir.mkdir(parents=True, exist_ok=True)

    shap_csv  = out_dir / "shap_importance.csv"
    shap_bar  = out_dir / "shap_bar_top20.png"
    shap_sum  = out_dir / "shap_summary.png"
    shap_meta = out_dir / "shap_meta.csv"

    # varsa atla
    if shap_csv.exists() and shap_bar.exists() and shap_sum.exists():
        print("⏩ Zaten var, atlandı:", name, flush=True)
        continue

    print("1) preprocess.transform(X_test) ...", flush=True)
    Xte_proc = pipe.named_steps["preprocess"].transform(X_test)
    print("   ✅ transform bitti. shape:", Xte_proc.shape, flush=True)

    model = pipe.named_steps["model"]
        # XGBoost özel-case: TreeExplainer bu ortamda base_score hatası veriyor
    if name == "XGBoost":
        print("2) XGBoost için TreeExplainer çalışmıyor -> shap.Explainer kullanılacak", flush=True)

        # Background (hız için) - train'den küçük bir örnek
        Xtr_proc = pipe.named_steps["preprocess"].transform(X_train)
        bg_n = 300
        idx_bg = rng.choice(Xtr_proc.shape[0], size=min(bg_n, Xtr_proc.shape[0]), replace=False)
        bg = Xtr_proc[idx_bg]

        # model proba fonksiyonu
        f = lambda x: model.predict_proba(x)[:, 1]

        explainer = shap.Explainer(f, bg)

        # full test için chunk ile ilerleme
        n, p = Xte_proc.shape
        sum_abs = np.zeros(p, dtype=np.float64)
        num_chunks = (n + CHUNK_SIZE - 1) // CHUNK_SIZE
        print(f"4) Chunked SHAP (XGBoost-Explainer): chunks={num_chunks}", flush=True)

        for i in range(num_chunks):
            a = i * CHUNK_SIZE
            b = min((i + 1) * CHUNK_SIZE, n)
            t_chunk = time.time()

            print(f"   ▶ chunk {i+1}/{num_chunks} rows {a}:{b} ...", flush=True)

            shap_exp = explainer(Xte_proc[a:b])      # shap.Explanation
            shap_mat = shap_exp.values               # (chunk, features)

            sum_abs += np.abs(shap_mat).sum(axis=0)

            dt = time.time() - t_chunk
            done = b
            rate = done / max(1e-9, (time.time() - t_model))
            eta = (n - done) / max(rate, 1e-9)
            print(f"   ✅ {dt:.2f}s | ilerleme: {done}/{n} | kalan: {eta/60:.1f} dk", flush=True)

        mean_abs = sum_abs / n
        df_shap = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs}) \
                    .sort_values("mean_abs_shap", ascending=False)

        df_shap.to_csv(shap_csv, index=False, encoding="utf-8")
        save_shap_bar(df_shap, shap_bar, f"{name} - SHAP Importance (Top 20) [full test, Explainer]")

        # summary plot (örneklem)
        plot_n = min(PLOT_SAMPLE, n)
        idx_plot = rng.choice(n, size=plot_n, replace=False)
        X_plot = Xte_proc[idx_plot]

        print(f"5) Summary plot (plot_n={plot_n}) ...", flush=True)
        shap_exp_plot = explainer(X_plot)
        shap_mat_plot = shap_exp_plot.values

        plt.figure()
        shap.summary_plot(shap_mat_plot, X_plot, feature_names=feature_cols, show=False)
        plt.tight_layout()
        plt.savefig(shap_sum, dpi=200)
        plt.close()

        meta = {
            "method": "shap.Explainer(f, bg) (full test, chunked)",
            "test_size": int(n),
            "chunk_size": int(CHUNK_SIZE),
            "plot_sample_n": int(plot_n),
            "bg_n": int(bg.shape[0]),
            "note": "TreeExplainer failed due to XGBoost base_score parsing issue."
        }
        pd.Series(meta).to_csv(shap_meta, encoding="utf-8")

        print(f"✅ {name} bitti. Süre: {(time.time()-t_model)/60:.1f} dk", flush=True)
        continue
    # -------------------------
    # TREE MODELLER
    # -------------------------
    if name in TREE_MODELS:
        print("2) TreeExplainer hazırlanıyor...", flush=True)
        try:
            explainer = shap.TreeExplainer(model)
        except Exception as e:
            # XGBoost base_score parse hatası vs. için fallback
            print("   ⚠️ TreeExplainer(model) başarısız -> data ile fallback:", str(e)[:120], "...", flush=True)
            explainer = shap.TreeExplainer(model, Xte_proc[:2000])

        print("3) warm-up (50 satır) ...", flush=True)
        _ = explainer.shap_values(Xte_proc[:50], approximate=True, check_additivity=False)
        print("   ✅ warm-up tamam", flush=True)

        n, p = Xte_proc.shape
        sum_abs = np.zeros(p, dtype=np.float64)
        num_chunks = (n + CHUNK_SIZE - 1) // CHUNK_SIZE
        print(f"4) Chunked SHAP: CHUNK_SIZE={CHUNK_SIZE}, chunks={num_chunks}", flush=True)

        for i in range(num_chunks):
            a = i * CHUNK_SIZE
            b = min((i + 1) * CHUNK_SIZE, n)
            t_chunk = time.time()

            print(f"   ▶ chunk {i+1}/{num_chunks} rows {a}:{b} ...", flush=True)

            shap_vals = explainer.shap_values(Xte_proc[a:b], approximate=True, check_additivity=False)
            shap_mat  = to_2d_shap(shap_vals)
            sum_abs  += np.abs(shap_mat).sum(axis=0)

            dt = time.time() - t_chunk
            done = b
            rate = done / max(1e-9, (time.time() - t_model))
            eta = (n - done) / max(rate, 1e-9)
            print(f"   ✅ {dt:.2f}s | ilerleme: {done}/{n} | kalan: {eta/60:.1f} dk", flush=True)

        mean_abs = sum_abs / n
        df_shap = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs})\
                    .sort_values("mean_abs_shap", ascending=False)

        df_shap.to_csv(shap_csv, index=False, encoding="utf-8")
        save_shap_bar(df_shap, shap_bar, f"{name} - SHAP Importance (Top 20) [full test, approx]")

        # summary plot (örneklem)
        plot_n = min(PLOT_SAMPLE, n)
        idx_plot = rng.choice(n, size=plot_n, replace=False)
        X_plot = Xte_proc[idx_plot]

        print(f"5) Summary plot (plot_n={plot_n}) ...", flush=True)
        shap_vals_plot = explainer.shap_values(X_plot, approximate=True, check_additivity=False)
        shap_mat_plot  = to_2d_shap(shap_vals_plot)

        plt.figure()
        shap.summary_plot(shap_mat_plot, X_plot, feature_names=feature_cols, show=False)
        plt.tight_layout()
        plt.savefig(shap_sum, dpi=200)
        plt.close()

        meta = {
            "method": "TreeExplainer (full test, chunked, approximate)",
            "test_size": int(n),
            "chunk_size": int(CHUNK_SIZE),
            "plot_sample_n": int(plot_n),
            "approximate": True,
            "check_additivity": False,
            "seed": int(SEED)
        }
        pd.Series(meta).to_csv(shap_meta, encoding="utf-8")

        print(f"✅ {name} bitti. Süre: {(time.time()-t_model)/60:.1f} dk", flush=True)

    # -------------------------
    # LOGISTIC REGRESSION
    # -------------------------
    else:
        # LogReg -> LinearExplainer
        print("2) LinearExplainer hazırlanıyor...", flush=True)

        # background (hız için)
        ntr = pipe.named_steps["preprocess"].transform(X_train).shape[0]
        # train'i tekrar transform etmeyelim diye:
        Xtr_proc = pipe.named_steps["preprocess"].transform(X_train)
        bg_n = min(20000, Xtr_proc.shape[0])
        idx_bg = rng.choice(Xtr_proc.shape[0], size=bg_n, replace=False)
        bg = Xtr_proc[idx_bg]

        explainer = shap.LinearExplainer(model, bg)

        print("3) SHAP values (full test) ...", flush=True)
        shap_vals = explainer.shap_values(Xte_proc)
        shap_mat  = to_2d_shap(shap_vals)

        mean_abs = np.abs(shap_mat).mean(axis=0)
        df_shap = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs})\
                    .sort_values("mean_abs_shap", ascending=False)

        df_shap.to_csv(shap_csv, index=False, encoding="utf-8")
        save_shap_bar(df_shap, shap_bar, f"{name} - SHAP Importance (Top 20) [full test]")

        plot_n = min(PLOT_SAMPLE, Xte_proc.shape[0])
        idx_plot = rng.choice(Xte_proc.shape[0], size=plot_n, replace=False)
        plt.figure()
        shap.summary_plot(shap_mat[idx_plot], Xte_proc[idx_plot], feature_names=feature_cols, show=False)
        plt.tight_layout()
        plt.savefig(shap_sum, dpi=200)
        plt.close()

        meta = {
            "method": "LinearExplainer (full test)",
            "test_size": int(Xte_proc.shape[0]),
            "background_n": int(bg_n),
            "seed": int(SEED)
        }
        pd.Series(meta).to_csv(shap_meta, encoding="utf-8")

        print(f"✅ {name} bitti. Süre: {(time.time()-t_model)/60:.1f} dk", flush=True)

print("\n✅ KNN hariç tüm modeller için SHAP global tamamlandı (full test).", flush=True)

✅ Full test üzerinde çalışıyoruz: X_test=(115984, 75), seed=42

🚀 SHAP FULL TEST: LogReg
1) preprocess.transform(X_test) ...
   ✅ transform bitti. shape: (115984, 75)
2) LinearExplainer hazırlanıyor...
3) SHAP values (full test) ...


C:\Users\elifo\AppData\Local\Temp\ipykernel_17764\4213363122.py:247: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_mat[idx_plot], Xte_proc[idx_plot], feature_names=feature_cols, show=False)


✅ LogReg bitti. Süre: 0.1 dk

🚀 SHAP FULL TEST: DecisionTree
1) preprocess.transform(X_test) ...
   ✅ transform bitti. shape: (115984, 75)
2) TreeExplainer hazırlanıyor...
3) warm-up (50 satır) ...
   ✅ warm-up tamam
4) Chunked SHAP: CHUNK_SIZE=2000, chunks=58
   ▶ chunk 1/58 rows 0:2000 ...
   ✅ 0.01s | ilerleme: 2000/115984 | kalan: 0.1 dk
   ▶ chunk 2/58 rows 2000:4000 ...
   ✅ 0.01s | ilerleme: 4000/115984 | kalan: 0.1 dk
   ▶ chunk 3/58 rows 4000:6000 ...
   ✅ 0.01s | ilerleme: 6000/115984 | kalan: 0.0 dk
   ▶ chunk 4/58 rows 6000:8000 ...
   ✅ 0.01s | ilerleme: 8000/115984 | kalan: 0.0 dk
   ▶ chunk 5/58 rows 8000:10000 ...
   ✅ 0.01s | ilerleme: 10000/115984 | kalan: 0.0 dk
   ▶ chunk 6/58 rows 10000:12000 ...
   ✅ 0.01s | ilerleme: 12000/115984 | kalan: 0.0 dk
   ▶ chunk 7/58 rows 12000:14000 ...
   ✅ 0.01s | ilerleme: 14000/115984 | kalan: 0.0 dk
   ▶ chunk 8/58 rows 14000:16000 ...
   ✅ 0.01s | ilerleme: 16000/115984 | kalan: 0.0 dk
   ▶ chunk 9/58 rows 16000:18000 ...
   ✅ 0

C:\Users\elifo\AppData\Local\Temp\ipykernel_17764\4213363122.py:198: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_mat_plot, X_plot, feature_names=feature_cols, show=False)


✅ DecisionTree bitti. Süre: 0.1 dk

🚀 SHAP FULL TEST: RandomForest
1) preprocess.transform(X_test) ...
   ✅ transform bitti. shape: (115984, 75)
2) TreeExplainer hazırlanıyor...
3) warm-up (50 satır) ...
   ✅ warm-up tamam
4) Chunked SHAP: CHUNK_SIZE=2000, chunks=58
   ▶ chunk 1/58 rows 0:2000 ...
   ✅ 0.80s | ilerleme: 2000/115984 | kalan: 1.8 dk
   ▶ chunk 2/58 rows 2000:4000 ...
   ✅ 0.73s | ilerleme: 4000/115984 | kalan: 1.2 dk
   ▶ chunk 3/58 rows 4000:6000 ...
   ✅ 0.84s | ilerleme: 6000/115984 | kalan: 1.1 dk
   ▶ chunk 4/58 rows 6000:8000 ...
   ✅ 0.76s | ilerleme: 8000/115984 | kalan: 0.9 dk
   ▶ chunk 5/58 rows 8000:10000 ...
   ✅ 0.72s | ilerleme: 10000/115984 | kalan: 0.9 dk
   ▶ chunk 6/58 rows 10000:12000 ...
   ✅ 0.69s | ilerleme: 12000/115984 | kalan: 0.8 dk
   ▶ chunk 7/58 rows 12000:14000 ...
   ✅ 0.75s | ilerleme: 14000/115984 | kalan: 0.8 dk
   ▶ chunk 8/58 rows 14000:16000 ...
   ✅ 0.77s | ilerleme: 16000/115984 | kalan: 0.7 dk
   ▶ chunk 9/58 rows 16000:18000 ...


...

PermutationExplainer explainer: 1985it [03:50,  8.24it/s]                                            


   ✅ 230.17s | ilerleme: 115984/115984 | kalan: 0.0 dk
5) Summary plot (plot_n=5000) ...


PermutationExplainer explainer: 5001it [09:56,  8.27it/s]                                            
C:\Users\elifo\AppData\Local\Temp\ipykernel_17764\4213363122.py:126: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_mat_plot, X_plot, feature_names=feature_cols, show=False)


✅ XGBoost bitti. Süre: 244.1 dk

🚀 SHAP FULL TEST: LightGBM
1) preprocess.transform(X_test) ...
   ✅ transform bitti. shape: (115984, 75)
2) TreeExplainer hazırlanıyor...
3) warm-up (50 satır) ...


AssertionError: approximate=True is not supported for LightGBM models!

In [1]:
#  kernel restart yaptım geri yükleme içim:
import os, random
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# -------- CONFIG --------
CONFIG = {
    "seed": 42,
    "test_size": 0.20,
    "label_col": "label",
    "url_col": "url",
    "output_dir": "outputs",
}

DATA_PATH = "~/Bitirme_Projesi_Dataset_Olusturma/Makale/final_lexical_dns_whois_cleaned.csv"

# -------- Seeds --------
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(CONFIG["seed"])

# -------- Paths --------
BASE_OUT = Path(CONFIG["output_dir"])
SHARED = BASE_OUT / "_shared"

# -------- Load dataset --------
df = pd.read_csv(DATA_PATH)
label_col = CONFIG["label_col"]
url_col = CONFIG["url_col"]

feature_cols = [c for c in df.columns if c not in [label_col, url_col]]
X = df[feature_cols].copy()
y = df[label_col].copy()

# -------- Load split indices --------
train_idx = pd.read_csv(SHARED / "split_indices_train.csv")["train_idx"].values
test_idx  = pd.read_csv(SHARED / "split_indices_test.csv")["test_idx"].values

X_train = X.iloc[train_idx].copy()
X_test  = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test  = y.iloc[test_idx].copy()

# -------- Preprocessors --------
tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# -------- Models (baseline Aşama 3 ile aynı) --------
base_models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=CONFIG["seed"]),
    "DecisionTree": DecisionTreeClassifier(random_state=CONFIG["seed"]),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=CONFIG["seed"], n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=CONFIG["seed"], eval_metric="logloss", n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=800, learning_rate=0.05, num_leaves=63,
        random_state=CONFIG["seed"], n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=800, learning_rate=0.05, depth=8,
        random_seed=CONFIG["seed"], verbose=False
    )
}

LINEAR_DISTANCE = {"LogReg", "KNN"}

# -------- Fit pipelines --------
pipelines = {}
for name, clf in base_models.items():
    print("Fitting:", name)
    pre = linear_preprocessor if name in LINEAR_DISTANCE else tree_preprocessor
    pipe = Pipeline([("preprocess", pre), ("model", clf)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

print("\n✅ Kernel reset sonrası SHAP için her şey hazır")
print("X_train:", X_train.shape, "X_test:", X_test.shape, "features:", len(feature_cols))


Fitting: LogReg
Fitting: DecisionTree
Fitting: KNN
Fitting: RandomForest
Fitting: XGBoost
Fitting: LightGBM
[LightGBM] [Info] Number of positive: 192677, number of negative: 271259
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055108 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6057
[LightGBM] [Info] Number of data points in the train set: 463936, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342059
[LightGBM] [Info] Start training from score -0.342059
Fitting: CatBoost

✅ Kernel reset sonrası SHAP için her şey hazır
X_train: (463936, 75) X_test: (115984, 75) features: 75


In [2]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

# =========================
# AYARLAR
# =========================
CHUNK_SIZE  = 2000   # daha sık çıktı için 1000 yapabilirsin
PLOT_SAMPLE = 5000
SEED        = CONFIG["seed"]

MODELS = ["LightGBM", "CatBoost"]
rng = np.random.RandomState(SEED)

def to_2d_shap(shap_vals):
    if isinstance(shap_vals, list):
        if len(shap_vals) >= 2:
            return np.asarray(shap_vals[1])  # class=1
        return np.asarray(shap_vals[-1])
    arr = np.asarray(shap_vals)
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3:
        if arr.shape[-1] >= 2:
            return arr[:, :, 1]
        return np.squeeze(arr, axis=-1)
    return np.squeeze(arr)

def save_shap_bar(df_shap, out_path, title, topn=20):
    top = df_shap.head(topn)
    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"][::-1], top["mean_abs_shap"][::-1])
    plt.title(title)
    plt.xlabel("mean(|SHAP|)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def tree_shap_values(explainer, X, model_name):
    # LightGBM approximate desteklemez; CatBoost'ta da approximate vermiyoruz.
    return explainer.shap_values(X, check_additivity=False)

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

print(f"✅ Full test: X_test={X_test.shape}, seed={SEED}", flush=True)

for name in MODELS:
    t_model = time.time()
    print("\n============================", flush=True)
    print(f"🚀 SHAP FULL TEST: {name}", flush=True)
    print("============================", flush=True)

    pipe = pipelines[name]
    out_dir = ensure_dir(BASE_OUT / name / "global_fulltest_approx")

    shap_csv  = out_dir / "shap_importance.csv"
    shap_bar  = out_dir / "shap_bar_top20.png"
    shap_sum  = out_dir / "shap_summary.png"
    shap_meta = out_dir / "shap_meta.csv"

    print("1) preprocess.transform(X_test) ...", flush=True)
    Xte_proc = pipe.named_steps["preprocess"].transform(X_test)
    print("   ✅ transform bitti. shape:", Xte_proc.shape, flush=True)

    model = pipe.named_steps["model"]

    print("2) TreeExplainer hazırlanıyor...", flush=True)
    explainer = shap.TreeExplainer(model)

    print("3) warm-up (50 satır) ...", flush=True)
    _ = tree_shap_values(explainer, Xte_proc[:50], name)
    print("   ✅ warm-up tamam", flush=True)

    n, p = Xte_proc.shape
    sum_abs = np.zeros(p, dtype=np.float64)
    num_chunks = (n + CHUNK_SIZE - 1) // CHUNK_SIZE
    print(f"4) Chunked SHAP: CHUNK_SIZE={CHUNK_SIZE}, chunks={num_chunks}", flush=True)

    for i in range(num_chunks):
        a = i * CHUNK_SIZE
        b = min((i + 1) * CHUNK_SIZE, n)
        t_chunk = time.time()

        print(f"   ▶ chunk {i+1}/{num_chunks} rows {a}:{b} ...", flush=True)

        shap_vals = tree_shap_values(explainer, Xte_proc[a:b], name)
        shap_mat  = to_2d_shap(shap_vals)
        sum_abs  += np.abs(shap_mat).sum(axis=0)

        dt = time.time() - t_chunk
        done = b
        rate = done / max(1e-9, (time.time() - t_model))
        eta = (n - done) / max(rate, 1e-9)
        print(f"   ✅ {dt:.2f}s | ilerleme: {done}/{n} | kalan: {eta/60:.1f} dk", flush=True)

    mean_abs = sum_abs / n
    df_shap = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs}) \
                .sort_values("mean_abs_shap", ascending=False)

    df_shap.to_csv(shap_csv, index=False, encoding="utf-8")
    save_shap_bar(df_shap, shap_bar, f"{name} - SHAP Importance (Top 20) [full test]")

    plot_n = min(PLOT_SAMPLE, n)
    idx_plot = rng.choice(n, size=plot_n, replace=False)
    X_plot = Xte_proc[idx_plot]

    print(f"5) Summary plot (plot_n={plot_n}) ...", flush=True)
    shap_vals_plot = tree_shap_values(explainer, X_plot, name)
    shap_mat_plot  = to_2d_shap(shap_vals_plot)

    plt.figure()
    shap.summary_plot(shap_mat_plot, X_plot, feature_names=feature_cols, show=False)
    plt.tight_layout()
    plt.savefig(shap_sum, dpi=200)
    plt.close()

    meta = {
        "model": name,
        "method": "TreeExplainer (chunked, full test)",
        "test_size": int(n),
        "chunk_size": int(CHUNK_SIZE),
        "plot_sample_n": int(plot_n),
        "check_additivity": False,
        "note": "LightGBM does not support approximate=True in SHAP TreeExplainer; CatBoost run without approximate for stability.",
        "seed": int(SEED)
    }
    pd.Series(meta).to_csv(shap_meta, encoding="utf-8")

    print(f"✅ {name} bitti. Süre: {(time.time()-t_model)/60:.1f} dk", flush=True)

print("\n✅ LightGBM + CatBoost için SHAP global tamamlandı.", flush=True)

✅ Full test: X_test=(115984, 75), seed=42

🚀 SHAP FULL TEST: LightGBM
1) preprocess.transform(X_test) ...
   ✅ transform bitti. shape: (115984, 75)
2) TreeExplainer hazırlanıyor...
3) warm-up (50 satır) ...
   ✅ warm-up tamam
4) Chunked SHAP: CHUNK_SIZE=2000, chunks=58
   ▶ chunk 1/58 rows 0:2000 ...


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


   ✅ 9.69s | ilerleme: 2000/115984 | kalan: 11.9 dk
   ▶ chunk 2/58 rows 2000:4000 ...
   ✅ 10.03s | ilerleme: 4000/115984 | kalan: 10.5 dk
   ▶ chunk 3/58 rows 4000:6000 ...
   ✅ 9.87s | ilerleme: 6000/115984 | kalan: 9.9 dk
   ▶ chunk 4/58 rows 6000:8000 ...
   ✅ 9.85s | ilerleme: 8000/115984 | kalan: 9.5 dk
   ▶ chunk 5/58 rows 8000:10000 ...
   ✅ 9.31s | ilerleme: 10000/115984 | kalan: 9.1 dk
   ▶ chunk 6/58 rows 10000:12000 ...
   ✅ 9.44s | ilerleme: 12000/115984 | kalan: 8.8 dk
   ▶ chunk 7/58 rows 12000:14000 ...
   ✅ 10.57s | ilerleme: 14000/115984 | kalan: 8.7 dk
   ▶ chunk 8/58 rows 14000:16000 ...
   ✅ 11.19s | ilerleme: 16000/115984 | kalan: 8.6 dk
   ▶ chunk 9/58 rows 16000:18000 ...
   ✅ 11.70s | ilerleme: 18000/115984 | kalan: 8.6 dk
   ▶ chunk 10/58 rows 18000:20000 ...
   ✅ 11.26s | ilerleme: 20000/115984 | kalan: 8.5 dk
   ▶ chunk 11/58 rows 20000:22000 ...
   ✅ 10.50s | ilerleme: 22000/115984 | kalan: 8.3 dk
   ▶ chunk 12/58 rows 22000:24000 ...
   ✅ 9.50s | ilerleme

C:\Users\elifo\AppData\Local\Temp\ipykernel_2696\3158991544.py:117: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_mat_plot, X_plot, feature_names=feature_cols, show=False)


✅ LightGBM bitti. Süre: 10.0 dk

🚀 SHAP FULL TEST: CatBoost
1) preprocess.transform(X_test) ...
   ✅ transform bitti. shape: (115984, 75)
2) TreeExplainer hazırlanıyor...
3) warm-up (50 satır) ...
   ✅ warm-up tamam
4) Chunked SHAP: CHUNK_SIZE=2000, chunks=58
   ▶ chunk 1/58 rows 0:2000 ...
   ✅ 13.25s | ilerleme: 2000/115984 | kalan: 16.2 dk
   ▶ chunk 2/58 rows 2000:4000 ...
   ✅ 13.40s | ilerleme: 4000/115984 | kalan: 14.2 dk
   ▶ chunk 3/58 rows 4000:6000 ...
   ✅ 13.33s | ilerleme: 6000/115984 | kalan: 13.4 dk
   ▶ chunk 4/58 rows 6000:8000 ...
   ✅ 13.12s | ilerleme: 8000/115984 | kalan: 12.8 dk
   ▶ chunk 5/58 rows 8000:10000 ...
   ✅ 13.32s | ilerleme: 10000/115984 | kalan: 12.4 dk
   ▶ chunk 6/58 rows 10000:12000 ...
   ✅ 13.52s | ilerleme: 12000/115984 | kalan: 12.1 dk
   ▶ chunk 7/58 rows 12000:14000 ...
   ✅ 13.14s | ilerleme: 14000/115984 | kalan: 11.8 dk
   ▶ chunk 8/58 rows 14000:16000 ...
   ✅ 13.45s | ilerleme: 16000/115984 | kalan: 11.5 dk
   ▶ chunk 9/58 rows 16000:1

C:\Users\elifo\AppData\Local\Temp\ipykernel_2696\3158991544.py:117: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_mat_plot, X_plot, feature_names=feature_cols, show=False)


✅ CatBoost bitti. Süre: 13.7 dk

✅ LightGBM + CatBoost için SHAP global tamamlandı.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# =========================================
# AYARLAR
# =========================================
MODELS = ["LogReg", "DecisionTree", "RandomForest", 
          "XGBoost", "LightGBM", "CatBoost"]

BASE = BASE_OUT  # senin mevcut output klasörün

print("🔎 SHAP tabloları toplanıyor...\n")

dfs = []

for model in MODELS:
    path = BASE / model / "global" / "shap_importance.csv"
    
    if not path.exists():
        print(f"⚠️ Bulunamadı: {model}")
        continue
        
    df = pd.read_csv(path)
    df = df.rename(columns={"mean_abs_shap": model})
    dfs.append(df[["feature", model]])

# =========================================
# TÜM MODELLERİ BİRLEŞTİR
# =========================================
df_all = dfs[0]

for df in dfs[1:]:
    df_all = df_all.merge(df, on="feature", how="outer")

df_all.fillna(0, inplace=True)

print("✅ Birleşik tablo oluşturuldu.")
print("Shape:", df_all.shape)

# =========================================
# ENSEMBLE ORTALAMA SHAP
# =========================================
df_all["ensemble_mean"] = df_all[MODELS].mean(axis=1)

# Rank hesapla
for model in MODELS:
    df_all[f"{model}_rank"] = df_all[model].rank(ascending=False)

df_all["ensemble_rank"] = df_all["ensemble_mean"].rank(ascending=False)

# =========================================
# ENSEMBLE TOP 20
# =========================================
df_sorted = df_all.sort_values("ensemble_mean", ascending=False)

top20 = df_sorted.head(20)

plt.figure(figsize=(10,8))
plt.barh(top20["feature"][::-1], top20["ensemble_mean"][::-1])
plt.title("Ensemble Mean SHAP Importance (Top 20)")
plt.xlabel("Mean(|SHAP|) - Ortalama (Tüm Modeller)")
plt.tight_layout()
plt.savefig(BASE / "_shared" / "ensemble_shap_top20.png", dpi=200)
plt.close()

print("📊 Ensemble Top20 grafiği kaydedildi.")

# =========================================
# STABILITY ANALİZİ
# =========================================
top10_counts = []

for feature in df_all["feature"]:
    count = 0
    for model in MODELS:
        rank = df_all.loc[df_all["feature"] == feature, f"{model}_rank"].values[0]
        if rank <= 10:
            count += 1
    top10_counts.append(count)

df_all["top10_model_count"] = top10_counts

df_all.to_csv(BASE / "_shared" / "shap_model_comparison_full.csv", index=False)

print("\n✅ Aşama 4.5 tamamlandı.")
print("Kaydedilen dosyalar:")
print(" - shap_model_comparison_full.csv")
print(" - ensemble_shap_top20.png")

🔎 SHAP tabloları toplanıyor...

✅ Birleşik tablo oluşturuldu.
Shape: (75, 7)
📊 Ensemble Top20 grafiği kaydedildi.

✅ Aşama 4.5 tamamlandı.
Kaydedilen dosyalar:
 - shap_model_comparison_full.csv
 - ensemble_shap_top20.png


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr

MODELS = ["LogReg", "DecisionTree", "RandomForest", 
          "XGBoost", "LightGBM", "CatBoost"]

BASE = BASE_OUT
results = []

print("🔎 SHAP vs Permutation karşılaştırması başlıyor...\n")

for model in MODELS:
    
    shap_path = BASE / model / "global" / "shap_importance.csv"
    perm_path = BASE / model / "global" / "perm_importance.csv"
    
    if not shap_path.exists() or not perm_path.exists():
        print(f"⚠️ Eksik dosya: {model}")
        continue
    
    shap_df = pd.read_csv(shap_path)
    perm_df = pd.read_csv(perm_path)
    
    # 🔎 importance kolonunu otomatik bul
    perm_cols = perm_df.columns
    importance_col = None
    
    for col in perm_cols:
        if "mean" in col.lower() or "importance" in col.lower():
            importance_col = col
            break
    
    if importance_col is None:
        print(f"❌ Importance kolonu bulunamadı: {model}")
        continue
    
    print(f"{model} → Perm kolon: {importance_col}")
    
    # Rank hesapla
    shap_df["shap_rank"] = shap_df["mean_abs_shap"].rank(ascending=False)
    perm_df["perm_rank"] = perm_df[importance_col].rank(ascending=False)
    
    merged = shap_df.merge(
        perm_df[["feature", "perm_rank"]],
        on="feature",
        how="inner"
    )
    
    corr, _ = spearmanr(merged["shap_rank"], merged["perm_rank"])
    
    results.append({
        "model": model,
        "spearman_rank_correlation": corr
    })
    
    print(f"{model} → Spearman ρ = {corr:.4f}")

df_corr = pd.DataFrame(results)
save_path = BASE / "_shared" / "shap_vs_permutation_correlation.csv"
df_corr.to_csv(save_path, index=False)

print("\n✅ SHAP vs Permutation analizi tamamlandı.")
print("Kaydedildi:", save_path)

🔎 SHAP vs Permutation karşılaştırması başlıyor...

LogReg → Perm kolon: importance_mean
LogReg → Spearman ρ = 0.9262
DecisionTree → Perm kolon: importance_mean
DecisionTree → Spearman ρ = 0.9678
RandomForest → Perm kolon: importance_mean
RandomForest → Spearman ρ = 0.8618
XGBoost → Perm kolon: importance_mean
XGBoost → Spearman ρ = 0.9409
LightGBM → Perm kolon: importance_mean
LightGBM → Spearman ρ = 0.9656
CatBoost → Perm kolon: importance_mean
CatBoost → Spearman ρ = 0.9431

✅ SHAP vs Permutation analizi tamamlandı.
Kaydedildi: outputs\_shared\shap_vs_permutation_correlation.csv


In [ ]:
//local analiz

In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = CONFIG["seed"]
rng = np.random.RandomState(SEED)

LOCAL_N_CORRECT = 30   # her model için doğru sınıftan kaç örnek
LOCAL_N_WRONG   = 30   # her model için yanlış sınıftan kaç örnek

LOCAL_MODELS = ["LogReg", "DecisionTree", "RandomForest", "LightGBM", "CatBoost"]  # ✅ XGBoost yok, ✅ KNN yok

out_shared = BASE_OUT / "_shared"
out_shared.mkdir(parents=True, exist_ok=True)

# Test tahminleriyle doğru/yanlış indexlerini çıkaracağız
local_index_records = []

for name in LOCAL_MODELS:
    pipe = pipelines[name]
    y_pred = pipe.predict(X_test)

    correct_idx = np.where(y_pred == y_test.values)[0]
    wrong_idx   = np.where(y_pred != y_test.values)[0]

    # doğru/yanlışlardan stratified seç (label korunacak)
    def stratified_pick(indices, n):
        if len(indices) == 0:
            return np.array([], dtype=int)
        # label'e göre ayır
        idx0 = indices[y_test.values[indices] == 0]
        idx1 = indices[y_test.values[indices] == 1]
        n0 = n // 2
        n1 = n - n0
        pick0 = rng.choice(idx0, size=min(n0, len(idx0)), replace=False) if len(idx0) else np.array([], dtype=int)
        pick1 = rng.choice(idx1, size=min(n1, len(idx1)), replace=False) if len(idx1) else np.array([], dtype=int)
        return np.concatenate([pick0, pick1])

    pick_correct = stratified_pick(correct_idx, LOCAL_N_CORRECT)
    pick_wrong   = stratified_pick(wrong_idx, LOCAL_N_WRONG)

    for ix in pick_correct:
        local_index_records.append({"model": name, "kind": "correct", "test_row_index": int(ix), "true_label": int(y_test.values[ix])})
    for ix in pick_wrong:
        local_index_records.append({"model": name, "kind": "wrong", "test_row_index": int(ix), "true_label": int(y_test.values[ix])})

df_local = pd.DataFrame(local_index_records).sort_values(["model","kind","true_label","test_row_index"])
save_path = out_shared / "local_samples.csv"
df_local.to_csv(save_path, index=False, encoding="utf-8")
print("✅ Local örnek listesi kaydedildi:", save_path)
df_local.head(10)

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ Local örnek listesi kaydedildi: outputs\_shared\local_samples.csv


,model,kind,test_row_index,true_label
244,CatBoost,correct,3740,0
251,CatBoost,correct,6780,0
253,CatBoost,correct,10031,0
245,CatBoost,correct,12265,0
242,CatBoost,correct,30597,0
254,CatBoost,correct,47933,0
243,CatBoost,correct,64020,0
240,CatBoost,correct,68612,0
252,CatBoost,correct,78336,0
247,CatBoost,correct,80302,0


In [11]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

SEED = CONFIG["seed"]
rng = np.random.RandomState(SEED)

# XGBoost yok, KNN yok
LOCAL_MODELS = ["LogReg", "DecisionTree", "RandomForest", "LightGBM", "CatBoost"]

MAX_WATERFALL_PER_GROUP = 30     # correct/wrong klasörüne en fazla kaç tane
MAX_DISPLAY = 20                # waterfall'da gösterilecek max feature
OUT_TAG = "local_v1"            # klasör adı etiket (karışmasın diye)

out_shared = BASE_OUT / "_shared"
df_local = pd.read_csv(out_shared / "local_samples.csv")

def to_2d_shap(shap_vals):
    if isinstance(shap_vals, list):
        if len(shap_vals) >= 2:
            return np.asarray(shap_vals[1])  # class=1
        return np.asarray(shap_vals[-1])
    arr = np.asarray(shap_vals)
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3:
        if arr.shape[-1] >= 2:
            return arr[:, :, 1]
        return np.squeeze(arr, axis=-1)
    return np.squeeze(arr)

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def eta_str(done, total, elapsed):
    if done <= 0:
        return "?"
    rate = done / max(elapsed, 1e-9)
    eta = (total - done) / max(rate, 1e-9)
    return f"{eta/60:.1f} dk"

print("✅ Local SHAP (XGBoost hariç) başlıyor...", flush=True)

for name in LOCAL_MODELS:
    t_model = time.time()
    print("\n============================", flush=True)
    print(f"🚀 LOCAL SHAP: {name}", flush=True)
    print("============================", flush=True)

    pipe = pipelines[name]
    out_dir = ensure_dir(BASE_OUT / name / OUT_TAG)

    # test preprocess
    print("1) preprocess.transform(X_test) ...", flush=True)
    Xte_proc = pipe.named_steps["preprocess"].transform(X_test)
    model = pipe.named_steps["model"]
    n, p = Xte_proc.shape
    print(f"   ✅ Xte_proc hazır: {Xte_proc.shape}", flush=True)

    # local indexler
    sub = df_local[df_local["model"] == name].copy()
    idx_correct = sub[sub["kind"] == "correct"]["test_row_index"].values[:MAX_WATERFALL_PER_GROUP]
    idx_wrong   = sub[sub["kind"] == "wrong"]["test_row_index"].values[:MAX_WATERFALL_PER_GROUP]

    # explainer
    if name == "LogReg":
        print("2) LinearExplainer hazırlanıyor (bg=20000) ...", flush=True)
        Xtr_proc = pipe.named_steps["preprocess"].transform(X_train)
        bg_n = min(20000, Xtr_proc.shape[0])
        bg = Xtr_proc[rng.choice(Xtr_proc.shape[0], size=bg_n, replace=False)]
        explainer = shap.LinearExplainer(model, bg)

        def shap_for_rows(idxs):
            X = Xte_proc[idxs]
            sv = explainer.shap_values(X)
            return X, to_2d_shap(sv), explainer.expected_value

    else:
        print("2) TreeExplainer hazırlanıyor ...", flush=True)
        explainer = shap.TreeExplainer(model)

        def shap_for_rows(idxs):
            X = Xte_proc[idxs]
            sv = explainer.shap_values(X, check_additivity=False)  # LGBM/CatBoost uyumlu
            return X, to_2d_shap(sv), explainer.expected_value

    def base_value_to_float(base):
        if isinstance(base, (list, np.ndarray)) and np.ndim(base) > 0:
            return float(base[-1])
        return float(base)

    def save_group(kind, idxs):
        folder = ensure_dir(out_dir / f"waterfall_{kind}")
        if len(idxs) == 0:
            print(f"   ⚠️ {kind}: örnek yok", flush=True)
            return

        print(f"3) {kind}: SHAP hesaplanıyor (n={len(idxs)})", flush=True)
        X_part, sv_part, base = shap_for_rows(idxs)
        base_val = base_value_to_float(base)

        total = len(idxs)
        for j, row_ix in enumerate(idxs, start=1):
            t0 = time.time()

            exp = shap.Explanation(
                values=sv_part[j-1],
                base_values=base_val,
                data=X_part[j-1],
                feature_names=feature_cols
            )
            plt.figure()
            shap.plots.waterfall(exp, max_display=MAX_DISPLAY, show=False)
            plt.tight_layout()
            plt.savefig(folder / f"{kind}_{int(row_ix)}.png", dpi=200)
            plt.close()

            elapsed = time.time() - t0
            total_elapsed = time.time() - t_model
            print(f"   ✅ {kind} {j}/{total} | {elapsed:.2f}s | ETA: {eta_str(j, total, total_elapsed)}", flush=True)

        print(f"   📁 Kaydedildi: {folder}", flush=True)

    save_group("correct", idx_correct)
    save_group("wrong", idx_wrong)

    # meta
    meta = {
        "model": name,
        "seed": int(SEED),
        "max_waterfall_per_group": int(MAX_WATERFALL_PER_GROUP),
        "max_display": int(MAX_DISPLAY),
        "note": "XGBoost local SHAP separate cell (last). KNN excluded from local SHAP (KernelSHAP sampling)."
    }
    pd.Series(meta).to_csv(out_dir / "local_meta.csv", encoding="utf-8")

    print(f"✅ {name} local bitti. Toplam süre: {(time.time()-t_model)/60:.1f} dk", flush=True)

print("\n✅ Local SHAP tamamlandı (XGBoost hariç).", flush=True)

✅ Local SHAP (XGBoost hariç) başlıyor...

🚀 LOCAL SHAP: LogReg
1) preprocess.transform(X_test) ...
   ✅ Xte_proc hazır: (115984, 75)
2) LinearExplainer hazırlanıyor (bg=20000) ...
3) correct: SHAP hesaplanıyor (n=30)
   ✅ correct 1/30 | 1.05s | ETA: 1.3 dk
   ✅ correct 2/30 | 1.06s | ETA: 0.9 dk
   ✅ correct 3/30 | 0.95s | ETA: 0.7 dk
   ✅ correct 4/30 | 0.83s | ETA: 0.6 dk
   ✅ correct 5/30 | 0.72s | ETA: 0.5 dk
   ✅ correct 6/30 | 0.68s | ETA: 0.5 dk
   ✅ correct 7/30 | 0.77s | ETA: 0.4 dk
   ✅ correct 8/30 | 0.80s | ETA: 0.4 dk
   ✅ correct 9/30 | 0.58s | ETA: 0.4 dk
   ✅ correct 10/30 | 0.67s | ETA: 0.3 dk
   ✅ correct 11/30 | 0.95s | ETA: 0.3 dk
   ✅ correct 12/30 | 0.91s | ETA: 0.3 dk
   ✅ correct 13/30 | 1.72s | ETA: 0.3 dk
   ✅ correct 14/30 | 1.61s | ETA: 0.3 dk
   ✅ correct 15/30 | 0.96s | ETA: 0.3 dk
   ✅ correct 16/30 | 0.61s | ETA: 0.2 dk
   ✅ correct 17/30 | 0.68s | ETA: 0.2 dk
   ✅ correct 18/30 | 0.94s | ETA: 0.2 dk
   ✅ correct 19/30 | 0.68s | ETA: 0.2 dk
   ✅ correct 

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


   ✅ correct 1/30 | 1.30s | ETA: 2.4 dk
   ✅ correct 2/30 | 1.04s | ETA: 1.4 dk
   ✅ correct 3/30 | 1.22s | ETA: 1.1 dk
   ✅ correct 4/30 | 1.11s | ETA: 0.9 dk
   ✅ correct 5/30 | 1.05s | ETA: 0.8 dk
   ✅ correct 6/30 | 1.08s | ETA: 0.7 dk
   ✅ correct 7/30 | 1.35s | ETA: 0.6 dk
   ✅ correct 8/30 | 0.93s | ETA: 0.6 dk
   ✅ correct 9/30 | 0.94s | ETA: 0.5 dk
   ✅ correct 10/30 | 0.95s | ETA: 0.5 dk
   ✅ correct 11/30 | 1.11s | ETA: 0.5 dk
   ✅ correct 12/30 | 1.14s | ETA: 0.4 dk
   ✅ correct 13/30 | 1.37s | ETA: 0.4 dk
   ✅ correct 14/30 | 1.02s | ETA: 0.4 dk
   ✅ correct 15/30 | 1.21s | ETA: 0.3 dk
   ✅ correct 16/30 | 1.26s | ETA: 0.3 dk
   ✅ correct 17/30 | 0.94s | ETA: 0.3 dk
   ✅ correct 18/30 | 0.96s | ETA: 0.3 dk
   ✅ correct 19/30 | 1.32s | ETA: 0.2 dk
   ✅ correct 20/30 | 0.94s | ETA: 0.2 dk
   ✅ correct 21/30 | 0.99s | ETA: 0.2 dk
   ✅ correct 22/30 | 0.97s | ETA: 0.2 dk
   ✅ correct 23/30 | 1.10s | ETA: 0.1 dk
   ✅ correct 24/30 | 0.95s | ETA: 0.1 dk
   ✅ correct 25/30 | 1.20

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


   ✅ wrong 1/30 | 1.47s | ETA: 18.2 dk
   ✅ wrong 2/30 | 1.05s | ETA: 9.1 dk
   ✅ wrong 3/30 | 1.12s | ETA: 6.0 dk
   ✅ wrong 4/30 | 0.95s | ETA: 4.4 dk
   ✅ wrong 5/30 | 1.11s | ETA: 3.5 dk
   ✅ wrong 6/30 | 0.97s | ETA: 2.9 dk
   ✅ wrong 7/30 | 1.19s | ETA: 2.4 dk
   ✅ wrong 8/30 | 0.93s | ETA: 2.1 dk
   ✅ wrong 9/30 | 1.04s | ETA: 1.8 dk
   ✅ wrong 10/30 | 1.17s | ETA: 1.6 dk
   ✅ wrong 11/30 | 1.08s | ETA: 1.4 dk
   ✅ wrong 12/30 | 1.05s | ETA: 1.2 dk
   ✅ wrong 13/30 | 1.43s | ETA: 1.1 dk
   ✅ wrong 14/30 | 1.14s | ETA: 1.0 dk
   ✅ wrong 15/30 | 1.12s | ETA: 0.9 dk
   ✅ wrong 16/30 | 0.93s | ETA: 0.8 dk
   ✅ wrong 17/30 | 1.12s | ETA: 0.7 dk
   ✅ wrong 18/30 | 1.02s | ETA: 0.6 dk
   ✅ wrong 19/30 | 1.35s | ETA: 0.6 dk
   ✅ wrong 20/30 | 1.05s | ETA: 0.5 dk
   ✅ wrong 21/30 | 1.18s | ETA: 0.4 dk
   ✅ wrong 22/30 | 0.99s | ETA: 0.4 dk
   ✅ wrong 23/30 | 1.00s | ETA: 0.3 dk
   ✅ wrong 24/30 | 0.90s | ETA: 0.3 dk
   ✅ wrong 25/30 | 1.33s | ETA: 0.2 dk
   ✅ wrong 26/30 | 0.96s | ETA: 0

In [13]:
import numpy as np
import pandas as pd

SEED = CONFIG["seed"]
rng = np.random.RandomState(SEED)

MODEL_NAME = "XGBoost"
LOCAL_N_CORRECT = 30
LOCAL_N_WRONG = 30

shared_path = BASE_OUT / "_shared" / "local_samples.csv"
df_local = pd.read_csv(shared_path)

# Eğer zaten XGBoost satırları varsa tekrar eklemeyelim
if (df_local["model"] == MODEL_NAME).any():
    print("✅ XGBoost örnekleri zaten var. Tekrar eklemedim.")
else:
    pipe = pipelines[MODEL_NAME]
    y_pred = pipe.predict(X_test)

    y_true = y_test.values if hasattr(y_test, "values") else np.asarray(y_test)

    correct_idx = np.where(y_pred == y_true)[0]
    wrong_idx   = np.where(y_pred != y_true)[0]

    def stratified_pick(indices, n):
        if len(indices) == 0:
            return np.array([], dtype=int)
        idx0 = indices[y_true[indices] == 0]
        idx1 = indices[y_true[indices] == 1]
        n0 = n // 2
        n1 = n - n0
        pick0 = rng.choice(idx0, size=min(n0, len(idx0)), replace=False) if len(idx0) else np.array([], dtype=int)
        pick1 = rng.choice(idx1, size=min(n1, len(idx1)), replace=False) if len(idx1) else np.array([], dtype=int)
        return np.concatenate([pick0, pick1])

    pick_correct = stratified_pick(correct_idx, LOCAL_N_CORRECT)
    pick_wrong   = stratified_pick(wrong_idx, LOCAL_N_WRONG)

    add_rows = []
    for ix in pick_correct:
        add_rows.append({"model": MODEL_NAME, "kind": "correct", "test_row_index": int(ix), "true_label": int(y_true[ix])})
    for ix in pick_wrong:
        add_rows.append({"model": MODEL_NAME, "kind": "wrong", "test_row_index": int(ix), "true_label": int(y_true[ix])})

    df_add = pd.DataFrame(add_rows)
    df_local2 = pd.concat([df_local, df_add], ignore_index=True)
    df_local2 = df_local2.sort_values(["model","kind","true_label","test_row_index"])

    df_local2.to_csv(shared_path, index=False, encoding="utf-8")

    print(f"✅ XGBoost örnekleri eklendi: correct={len(pick_correct)}, wrong={len(pick_wrong)}")
    print("📄 Güncellendi:", shared_path)

# Kontrol: XGBoost kaç satır var?
df_check = pd.read_csv(shared_path)
print("XGBoost satır sayısı:", (df_check["model"] == MODEL_NAME).sum())
df_check[df_check["model"] == MODEL_NAME].head(10)

✅ XGBoost örnekleri eklendi: correct=30, wrong=30
📄 Güncellendi: outputs\_shared\local_samples.csv
XGBoost satır sayısı: 60


,model,kind,test_row_index,true_label
300,XGBoost,correct,5871,0
301,XGBoost,correct,7098,0
302,XGBoost,correct,18067,0
303,XGBoost,correct,26326,0
304,XGBoost,correct,28447,0
305,XGBoost,correct,33618,0
306,XGBoost,correct,36939,0
307,XGBoost,correct,47754,0
308,XGBoost,correct,64558,0
309,XGBoost,correct,68839,0


In [15]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

SEED = CONFIG["seed"]
rng = np.random.RandomState(SEED)

MODEL_NAME = "XGBoost"
OUT_TAG = "local_v1"

# XGBoost local için güvenli sınırlar (istersen arttırırsın)
MAX_CORRECT = 30
MAX_WRONG   = 30
MAX_DISPLAY = 20

# Permutation/Explainer background boyutu (küçük tut -> hız)
BG_N = 200

out_shared = BASE_OUT / "_shared"
df_local = pd.read_csv(out_shared / "local_samples.csv")

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def eta_str(done, total, elapsed):
    if done <= 0:
        return "?"
    rate = done / max(elapsed, 1e-9)
    eta = (total - done) / max(rate, 1e-9)
    return f"{eta/60:.1f} dk"

print("\n============================", flush=True)
print("🚀 LOCAL SHAP (EN SON): XGBoost", flush=True)
print("============================", flush=True)

pipe = pipelines[MODEL_NAME]
out_dir = ensure_dir(BASE_OUT / MODEL_NAME / OUT_TAG)
folder_correct = ensure_dir(out_dir / "waterfall_correct")
folder_wrong   = ensure_dir(out_dir / "waterfall_wrong")

print("1) preprocess.transform(X_test) ...", flush=True)
Xte_proc = pipe.named_steps["preprocess"].transform(X_test)
model = pipe.named_steps["model"]
print("   ✅ Xte_proc:", Xte_proc.shape, flush=True)

print("2) preprocess.transform(X_train) (BG için) ...", flush=True)
Xtr_proc = pipe.named_steps["preprocess"].transform(X_train)
bg = Xtr_proc[rng.choice(Xtr_proc.shape[0], size=min(BG_N, Xtr_proc.shape[0]), replace=False)]
print("   ✅ BG hazır:", bg.shape, flush=True)

# Local örnekler
sub = df_local[df_local["model"] == MODEL_NAME].copy()
idx_correct = sub[sub["kind"] == "correct"]["test_row_index"].values[:MAX_CORRECT]
idx_wrong   = sub[sub["kind"] == "wrong"]["test_row_index"].values[:MAX_WRONG]

# XGBoost için SHAP: model-agnostic
f = lambda x: model.predict_proba(x)[:, 1]
print("3) shap.Explainer(f, bg) hazırlanıyor (model-agnostic) ...", flush=True)
explainer = shap.Explainer(f, bg)   # büyük olasılıkla PermutationExplainer seçecek
print("   ✅ Explainer hazır:", type(explainer).__name__, flush=True)

def explain_one(row_ix):
    x = Xte_proc[row_ix:row_ix+1]
    exp = explainer(x)  # shap.Explanation
    return exp

def save_one(exp, out_path):
    # exp.values: (1, n_features) -> squeeze
    e = shap.Explanation(
        values=np.squeeze(exp.values, axis=0),
        base_values=float(np.squeeze(exp.base_values)),
        data=np.squeeze(exp.data, axis=0),
        feature_names=feature_cols
    )
    plt.figure()
    shap.plots.waterfall(e, max_display=MAX_DISPLAY, show=False)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

t_all = time.time()

# correct
print(f"4) correct örnekler: n={len(idx_correct)}", flush=True)
for j, row_ix in enumerate(idx_correct, start=1):
    t0 = time.time()
    exp = explain_one(int(row_ix))
    save_one(exp, folder_correct / f"correct_{int(row_ix)}.png")
    elapsed = time.time() - t0
    total_elapsed = time.time() - t_all
    print(f"   ✅ correct {j}/{len(idx_correct)} | {elapsed:.1f}s | ETA: {eta_str(j, len(idx_correct), total_elapsed)}", flush=True)

# wrong
print(f"5) wrong örnekler: n={len(idx_wrong)}", flush=True)
t_wrong = time.time()
for j, row_ix in enumerate(idx_wrong, start=1):
    t0 = time.time()
    exp = explain_one(int(row_ix))
    save_one(exp, folder_wrong / f"wrong_{int(row_ix)}.png")
    elapsed = time.time() - t0
    total_elapsed = time.time() - t_wrong
    print(f"   ✅ wrong {j}/{len(idx_wrong)} | {elapsed:.1f}s | ETA: {eta_str(j, len(idx_wrong), total_elapsed)}", flush=True)

meta = {
    "model": MODEL_NAME,
    "seed": int(SEED),
    "bg_n": int(bg.shape[0]),
    "max_correct": int(len(idx_correct)),
    "max_wrong": int(len(idx_wrong)),
    "explainer_type": type(explainer).__name__,
    "note": "TreeExplainer failed in this environment (base_score parsing). Used model-agnostic shap.Explainer(f,bg), typically PermutationExplainer."
}
pd.Series(meta).to_csv(out_dir / "local_meta.csv", encoding="utf-8")

print("\n✅ XGBoost local SHAP bitti.")
print("📁 Çıktı klasörü:", out_dir)


🚀 LOCAL SHAP (EN SON): XGBoost
1) preprocess.transform(X_test) ...
   ✅ Xte_proc: (115984, 75)
2) preprocess.transform(X_train) (BG için) ...
   ✅ BG hazır: (200, 75)
3) shap.Explainer(f, bg) hazırlanıyor (model-agnostic) ...
   ✅ Explainer hazır: PermutationExplainer
4) correct örnekler: n=30
   ✅ correct 1/30 | 1.0s | ETA: 0.5 dk
   ✅ correct 2/30 | 0.7s | ETA: 0.4 dk
   ✅ correct 3/30 | 0.7s | ETA: 0.4 dk
   ✅ correct 4/30 | 0.7s | ETA: 0.3 dk
   ✅ correct 5/30 | 0.8s | ETA: 0.3 dk
   ✅ correct 6/30 | 1.0s | ETA: 0.3 dk
   ✅ correct 7/30 | 0.7s | ETA: 0.3 dk
   ✅ correct 8/30 | 0.7s | ETA: 0.3 dk
   ✅ correct 9/30 | 0.7s | ETA: 0.3 dk
   ✅ correct 10/30 | 0.7s | ETA: 0.3 dk
   ✅ correct 11/30 | 0.8s | ETA: 0.2 dk
   ✅ correct 12/30 | 0.7s | ETA: 0.2 dk
   ✅ correct 13/30 | 1.0s | ETA: 0.2 dk
   ✅ correct 14/30 | 0.7s | ETA: 0.2 dk
   ✅ correct 15/30 | 0.7s | ETA: 0.2 dk
   ✅ correct 16/30 | 0.8s | ETA: 0.2 dk
   ✅ correct 17/30 | 0.7s | ETA: 0.2 dk
   ✅ correct 18/30 | 0.8s | ETA: 

In [17]:
!pip install lime
!pip install scikit-learn


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached scikit_image-0.25.2-cp310-cp310-win_amd64.whl.metadata (14 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached tifffile-2025.5.10-py3-none-any.whl.metadata (31 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
Using cached scikit_image-0.25.2-cp310-cp310-win_amd64.whl (12.8 MB)
Using cached lazy_loader-0.4-py3-none-any.whl (12 kB)
Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
Using cached tifffile-2025.5.10-py3-none-any.whl (226 kB)
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283918 sha256=19deb72a6170a0edecc82ea754779ba3b70e2390bec34e38a5918997e5366a6e
  Stored in directory: c:\users\elifo\appdata\local\pip\cache\wheels\fd\a2\af\9ac0a1a85a27f314a06b39e1f492bee1547d52549a4606ed89
Successfully built lime

   ---------------------------------------- 0/6 [tifffile]
   ---------------------

  DEPRECATION: Building 'lime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'lime'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [18]:
# !pip install lime
# !pip install scikit-learn

import numpy as np
import pandas as pd
from pathlib import Path
import time

from lime.lime_tabular import LimeTabularExplainer

SEED = CONFIG["seed"]
rng = np.random.RandomState(SEED)

# KNN local yok (senin kararın)
# XGBoost local var (PermutationExplainer ile yaptın) -> LIME'da da yapabiliriz (model-agnostic)
LIME_MODELS = ["LogReg", "DecisionTree", "RandomForest", "XGBoost", "LightGBM", "CatBoost"]

MAX_PER_GROUP = 30     # correct/wrong max kaç örnek açıklansın
NUM_FEATURES  = 15     # LIME açıklamasında gösterilecek feature sayısı
N_SAMPLES     = 2000   # LIME perturbed sample sayısı (hız/kalite dengesi)

out_shared = BASE_OUT / "_shared"
df_local = pd.read_csv(out_shared / "local_samples.csv")

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def eta_str(done, total, elapsed):
    if done <= 0:
        return "?"
    rate = done / max(elapsed, 1e-9)
    eta = (total - done) / max(rate, 1e-9)
    return f"{eta/60:.1f} dk"

print("✅ Local LIME başlıyor...", flush=True)

for name in LIME_MODELS:
    t_model = time.time()
    print("\n============================", flush=True)
    print(f"🍋 LIME LOCAL: {name}", flush=True)
    print("============================", flush=True)

    pipe = pipelines[name]
    out_dir = ensure_dir(BASE_OUT / name / "lime_local_v1")
    out_correct = ensure_dir(out_dir / "correct_html")
    out_wrong   = ensure_dir(out_dir / "wrong_html")

    # preprocess
    print("1) preprocess.transform(X_train/X_test) ...", flush=True)
    Xtr_proc = pipe.named_steps["preprocess"].transform(X_train)
    Xte_proc = pipe.named_steps["preprocess"].transform(X_test)
    ytr = y_train.values if hasattr(y_train, "values") else np.asarray(y_train)

    # LIME, numeric space üzerinde çalışacak (preprocess sonrası)
    explainer = LimeTabularExplainer(
        training_data=np.array(Xtr_proc),
        training_labels=np.array(ytr),
        feature_names=feature_cols,
        class_names=["legit", "phishing"],
        discretize_continuous=True,  # LIME için genelde daha stabil
        mode="classification",
        random_state=SEED
    )

    model = pipe.named_steps["model"]

    # predict_proba fonksiyonu (LIME bunu ister)
    def predict_fn(x):
        return model.predict_proba(x)

    # seçilen örnekler
    sub = df_local[df_local["model"] == name].copy()
    idx_correct = sub[sub["kind"] == "correct"]["test_row_index"].values[:MAX_PER_GROUP]
    idx_wrong   = sub[sub["kind"] == "wrong"]["test_row_index"].values[:MAX_PER_GROUP]

    def run_group(kind, idxs, out_folder):
        if len(idxs) == 0:
            print(f"   ⚠️ {kind}: örnek yok", flush=True)
            return

        print(f"2) {kind} için LIME açıklamaları (n={len(idxs)})", flush=True)
        total = len(idxs)
        for j, row_ix in enumerate(idxs, start=1):
            t0 = time.time()
            x = np.array(Xte_proc[int(row_ix)])

            exp = explainer.explain_instance(
                data_row=x,
                predict_fn=predict_fn,
                num_features=NUM_FEATURES,
                num_samples=N_SAMPLES
            )

            html_path = out_folder / f"{kind}_{int(row_ix)}.html"
            exp.save_to_file(str(html_path))

            elapsed = time.time() - t0
            total_elapsed = time.time() - t_model
            print(f"   ✅ {kind} {j}/{total} | {elapsed:.1f}s | ETA: {eta_str(j, total, total_elapsed)}", flush=True)

        print(f"   📁 Kaydedildi: {out_folder}", flush=True)

    run_group("correct", idx_correct, out_correct)
    run_group("wrong", idx_wrong, out_wrong)

    meta = {
        "model": name,
        "seed": int(SEED),
        "num_features_shown": int(NUM_FEATURES),
        "num_samples": int(N_SAMPLES),
        "discretize_continuous": True,
        "note": "LIME computed on preprocessed numeric feature space to keep pipeline consistent across models."
    }
    pd.Series(meta).to_csv(out_dir / "lime_meta.csv", encoding="utf-8")

    print(f"✅ {name} LIME bitti. Süre: {(time.time()-t_model)/60:.1f} dk", flush=True)

print("\n✅ Aşama 5.3 (Local LIME) tamamlandı.", flush=True)

✅ Local LIME başlıyor...

🍋 LIME LOCAL: LogReg
1) preprocess.transform(X_train/X_test) ...
2) correct için LIME açıklamaları (n=30)
   ✅ correct 1/30 | 0.6s | ETA: 6.7 dk
   ✅ correct 2/30 | 0.2s | ETA: 3.3 dk
   ✅ correct 3/30 | 0.2s | ETA: 2.1 dk
   ✅ correct 4/30 | 0.2s | ETA: 1.6 dk
   ✅ correct 5/30 | 0.2s | ETA: 1.2 dk
   ✅ correct 6/30 | 0.2s | ETA: 1.0 dk
   ✅ correct 7/30 | 0.2s | ETA: 0.8 dk
   ✅ correct 8/30 | 0.2s | ETA: 0.7 dk
   ✅ correct 9/30 | 0.2s | ETA: 0.6 dk
   ✅ correct 10/30 | 0.2s | ETA: 0.5 dk
   ✅ correct 11/30 | 0.1s | ETA: 0.4 dk
   ✅ correct 12/30 | 0.1s | ETA: 0.4 dk
   ✅ correct 13/30 | 0.1s | ETA: 0.3 dk
   ✅ correct 14/30 | 0.1s | ETA: 0.3 dk
   ✅ correct 15/30 | 0.1s | ETA: 0.3 dk
   ✅ correct 16/30 | 0.1s | ETA: 0.2 dk
   ✅ correct 17/30 | 0.1s | ETA: 0.2 dk
   ✅ correct 18/30 | 0.1s | ETA: 0.2 dk
   ✅ correct 19/30 | 0.1s | ETA: 0.2 dk
   ✅ correct 20/30 | 0.1s | ETA: 0.1 dk
   ✅ correct 21/30 | 0.1s | ETA: 0.1 dk
   ✅ correct 22/30 | 0.1s | ETA: 0.1 

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ✅ correct 2/30 | 0.2s | ETA: 2.2 dk
   ✅ correct 3/30 | 0.2s | ETA: 1.4 dk


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ✅ correct 4/30 | 0.2s | ETA: 1.0 dk
   ✅ correct 5/30 | 0.2s | ETA: 0.8 dk


...

   ✅ wrong 28/30 | 0.2s | ETA: 0.0 dk


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ✅ wrong 29/30 | 0.2s | ETA: 0.0 dk
   ✅ wrong 30/30 | 0.2s | ETA: 0.0 dk
   📁 Kaydedildi: outputs\LightGBM\lime_local_v1\wrong_html
✅ LightGBM LIME bitti. Süre: 0.3 dk

🍋 LIME LOCAL: CatBoost
1) preprocess.transform(X_train/X_test) ...


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


2) correct için LIME açıklamaları (n=30)
   ✅ correct 1/30 | 0.2s | ETA: 5.1 dk
   ✅ correct 2/30 | 0.2s | ETA: 2.5 dk
   ✅ correct 3/30 | 0.2s | ETA: 1.7 dk
   ✅ correct 4/30 | 0.2s | ETA: 1.2 dk
   ✅ correct 5/30 | 0.2s | ETA: 1.0 dk
   ✅ correct 6/30 | 0.2s | ETA: 0.8 dk
   ✅ correct 7/30 | 0.2s | ETA: 0.6 dk
   ✅ correct 8/30 | 0.3s | ETA: 0.6 dk
   ✅ correct 9/30 | 0.3s | ETA: 0.5 dk
   ✅ correct 10/30 | 0.2s | ETA: 0.4 dk
   ✅ correct 11/30 | 0.1s | ETA: 0.4 dk
   ✅ correct 12/30 | 0.1s | ETA: 0.3 dk
   ✅ correct 13/30 | 0.2s | ETA: 0.3 dk
   ✅ correct 14/30 | 0.1s | ETA: 0.3 dk
   ✅ correct 15/30 | 0.2s | ETA: 0.2 dk
   ✅ correct 16/30 | 0.2s | ETA: 0.2 dk
   ✅ correct 17/30 | 0.2s | ETA: 0.2 dk
   ✅ correct 18/30 | 0.2s | ETA: 0.2 dk
   ✅ correct 19/30 | 0.1s | ETA: 0.1 dk
   ✅ correct 20/30 | 0.1s | ETA: 0.1 dk
   ✅ correct 21/30 | 0.1s | ETA: 0.1 dk
   ✅ correct 22/30 | 0.2s | ETA: 0.1 dk
   ✅ correct 23/30 | 0.1s | ETA: 0.1 dk
   ✅ correct 24/30 | 0.2s | ETA: 0.1 dk
   ✅ cor

In [20]:
# !pip install lime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

from lime.lime_tabular import LimeTabularExplainer

SEED = CONFIG["seed"]
rng = np.random.RandomState(SEED)

LIME_MODELS = ["LogReg", "DecisionTree", "RandomForest", "XGBoost", "LightGBM", "CatBoost"]  # KNN yok
MAX_PER_GROUP = 30
NUM_FEATURES  = 15
N_SAMPLES     = 2000

OUT_TAG = "lime_local_v1"

out_shared = BASE_OUT / "_shared"
df_local = pd.read_csv(out_shared / "local_samples.csv")

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def eta_str(done, total, elapsed):
    if done <= 0:
        return "?"
    rate = done / max(elapsed, 1e-9)
    eta = (total - done) / max(rate, 1e-9)
    return f"{eta/60:.1f} dk"

rows = []
print("✅ Local LIME (PNG + CSV) başlıyor...", flush=True)

for name in LIME_MODELS:
    t_model = time.time()
    print("\n============================", flush=True)
    print(f"🍋 LIME LOCAL: {name}", flush=True)
    print("============================", flush=True)

    pipe = pipelines[name]
    out_dir = ensure_dir(BASE_OUT / name / OUT_TAG)
    out_correct = ensure_dir(out_dir / "correct_png")
    out_wrong   = ensure_dir(out_dir / "wrong_png")

    print("1) preprocess.transform(X_train/X_test) ...", flush=True)
    Xtr_proc = pipe.named_steps["preprocess"].transform(X_train)
    Xte_proc = pipe.named_steps["preprocess"].transform(X_test)

    ytr = y_train.values if hasattr(y_train, "values") else np.asarray(y_train)
    yte = y_test.values if hasattr(y_test, "values") else np.asarray(y_test)

    model = pipe.named_steps["model"]
    def predict_fn(x):
        return model.predict_proba(x)

    explainer = LimeTabularExplainer(
        training_data=np.array(Xtr_proc),
        training_labels=np.array(ytr),
        feature_names=feature_cols,
        class_names=["legit", "phishing"],
        discretize_continuous=True,
        mode="classification",
        random_state=SEED
    )

    sub = df_local[df_local["model"] == name].copy()
    idx_correct = sub[sub["kind"] == "correct"]["test_row_index"].values[:MAX_PER_GROUP]
    idx_wrong   = sub[sub["kind"] == "wrong"]["test_row_index"].values[:MAX_PER_GROUP]

    def choose_label_for_plot(exp, pred_label):
        # exp.local_exp dict'inde hangi label'lar var?
        available = list(exp.local_exp.keys())
        if pred_label in available:
            return pred_label
        # yoksa ilk available label
        return available[0] if len(available) else pred_label

    def run_group(kind, idxs, out_folder):
        if len(idxs) == 0:
            print(f"   ⚠️ {kind}: örnek yok", flush=True)
            return

        print(f"2) {kind} için LIME (n={len(idxs)})", flush=True)
        total = len(idxs)

        for j, row_ix in enumerate(idxs, start=1):
            t0 = time.time()
            row_ix = int(row_ix)

            x = np.array(Xte_proc[row_ix])
            proba = predict_fn(Xte_proc[row_ix:row_ix+1])[0]
            pred_label = int(np.argmax(proba))
            true_label = int(yte[row_ix])

            # ✅ Kritik fix: top_labels=2 -> her iki sınıf için de exp üret
            exp = explainer.explain_instance(
                data_row=x,
                predict_fn=predict_fn,
                num_features=NUM_FEATURES,
                num_samples=N_SAMPLES,
                top_labels=2
            )

            plot_label = choose_label_for_plot(exp, pred_label)

            # PNG
            fig = exp.as_pyplot_figure(label=plot_label)
            fig.suptitle(
                f"{name} | {kind} | row={row_ix} | true={true_label} pred={pred_label} | plot_label={plot_label}",
                fontsize=10
            )
            plt.tight_layout()
            fig_path = out_folder / f"{kind}_{row_ix}.png"
            fig.savefig(fig_path, dpi=200)
            plt.close(fig)

            # CSV rows
            # açıklamayı hangi label ile çizdiysek, CSV'ye de onu basalım (tutarlı)
            for feat_desc, weight in exp.as_list(label=plot_label):
                rows.append({
                    "model": name,
                    "kind": kind,
                    "test_row_index": row_ix,
                    "true_label": true_label,
                    "pred_label": pred_label,
                    "pred_proba_phishing": float(proba[1]),
                    "class_explained": int(plot_label),
                    "feature_desc": feat_desc,
                    "weight": float(weight)
                })

            elapsed = time.time() - t0
            total_elapsed = time.time() - t_model
            print(f"   ✅ {kind} {j}/{total} | {elapsed:.1f}s | ETA: {eta_str(j, total, total_elapsed)}", flush=True)

        print(f"   📁 PNG kaydedildi: {out_folder}", flush=True)

    run_group("correct", idx_correct, out_correct)
    run_group("wrong", idx_wrong, out_wrong)

    meta = {
        "model": name,
        "seed": int(SEED),
        "num_features_shown": int(NUM_FEATURES),
        "num_samples": int(N_SAMPLES),
        "discretize_continuous": True,
        "top_labels": 2,
        "output": "PNG + shared CSV",
        "note": "top_labels=2 used to avoid missing class explanations (KeyError)."
    }
    pd.Series(meta).to_csv(out_dir / "lime_meta.csv", encoding="utf-8")

    print(f"✅ {name} LIME bitti. Süre: {(time.time()-t_model)/60:.1f} dk", flush=True)

df_lime = pd.DataFrame(rows)
save_csv = BASE_OUT / "_shared" / "lime_local_explanations_long.csv"
df_lime.to_csv(save_csv, index=False, encoding="utf-8")

print("\n✅ Aşama 5.3 tamamlandı.")
print("📄 LIME CSV:", save_csv)
print("🖼️ PNG'ler: outputs/{Model}/lime_local_v1/{correct_png,wrong_png}/")
df_lime.head()

✅ Local LIME (PNG + CSV) başlıyor...

🍋 LIME LOCAL: LogReg
1) preprocess.transform(X_train/X_test) ...
2) correct için LIME (n=30)
   ✅ correct 1/30 | 0.4s | ETA: 5.8 dk
   ✅ correct 2/30 | 0.3s | ETA: 2.9 dk
   ✅ correct 3/30 | 0.3s | ETA: 1.9 dk
   ✅ correct 4/30 | 0.3s | ETA: 1.4 dk
   ✅ correct 5/30 | 0.4s | ETA: 1.1 dk
   ✅ correct 6/30 | 0.3s | ETA: 0.9 dk
   ✅ correct 7/30 | 0.3s | ETA: 0.8 dk
   ✅ correct 8/30 | 0.3s | ETA: 0.7 dk
   ✅ correct 9/30 | 0.3s | ETA: 0.6 dk
   ✅ correct 10/30 | 0.4s | ETA: 0.5 dk
   ✅ correct 11/30 | 0.4s | ETA: 0.4 dk
   ✅ correct 12/30 | 0.3s | ETA: 0.4 dk
   ✅ correct 13/30 | 1.0s | ETA: 0.4 dk
   ✅ correct 14/30 | 0.3s | ETA: 0.3 dk
   ✅ correct 15/30 | 0.3s | ETA: 0.3 dk
   ✅ correct 16/30 | 0.4s | ETA: 0.3 dk
   ✅ correct 17/30 | 0.3s | ETA: 0.2 dk
   ✅ correct 18/30 | 0.3s | ETA: 0.2 dk
   ✅ correct 19/30 | 0.4s | ETA: 0.2 dk
   ✅ correct 20/30 | 0.3s | ETA: 0.2 dk
   ✅ correct 21/30 | 0.3s | ETA: 0.1 dk
   ✅ correct 22/30 | 0.3s | ETA: 0.1 d

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ✅ correct 1/30 | 0.5s | ETA: 6.2 dk


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ✅ correct 2/30 | 0.4s | ETA: 3.1 dk


...

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ✅ wrong 29/30 | 0.5s | ETA: 0.0 dk


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   ✅ wrong 30/30 | 0.5s | ETA: 0.0 dk
   📁 PNG kaydedildi: outputs\LightGBM\lime_local_v1\wrong_png
✅ LightGBM LIME bitti. Süre: 0.7 dk

🍋 LIME LOCAL: CatBoost
1) preprocess.transform(X_train/X_test) ...
2) correct için LIME (n=30)
   ✅ correct 1/30 | 0.4s | ETA: 5.0 dk
   ✅ correct 2/30 | 0.4s | ETA: 2.5 dk
   ✅ correct 3/30 | 0.4s | ETA: 1.7 dk
   ✅ correct 4/30 | 0.5s | ETA: 1.3 dk
   ✅ correct 5/30 | 0.6s | ETA: 1.0 dk
   ✅ correct 6/30 | 0.4s | ETA: 0.9 dk
   ✅ correct 7/30 | 0.4s | ETA: 0.7 dk
   ✅ correct 8/30 | 0.4s | ETA: 0.6 dk
   ✅ correct 9/30 | 0.3s | ETA: 0.5 dk
   ✅ correct 10/30 | 0.3s | ETA: 0.5 dk
   ✅ correct 11/30 | 0.4s | ETA: 0.4 dk
   ✅ correct 12/30 | 0.4s | ETA: 0.4 dk
   ✅ correct 13/30 | 0.4s | ETA: 0.3 dk
   ✅ correct 14/30 | 0.5s | ETA: 0.3 dk
   ✅ correct 15/30 | 0.4s | ETA: 0.3 dk
   ✅ correct 16/30 | 0.6s | ETA: 0.2 dk
   ✅ correct 17/30 | 0.3s | ETA: 0.2 dk
   ✅ correct 18/30 | 0.4s | ETA: 0.2 dk
   ✅ correct 19/30 | 0.4s | ETA: 0.2 dk
   ✅ correct 20/3

,model,kind,test_row_index,true_label,pred_label,pred_proba_phishing,class_explained,feature_desc,weight
0,LogReg,correct,1251,0,0,0.001571,0,hostname_length > 0.21,-0.423057
1,LogReg,correct,1251,0,0,0.001571,0,starts_with_ip <= -0.11,0.298548
2,LogReg,correct,1251,0,0,0.001571,0,query_entropy <= -0.29,-0.291029
3,LogReg,correct,1251,0,0,0.001571,0,num_slash > 0.03,0.283437
4,LogReg,correct,1251,0,0,0.001571,0,unusual_double_slash <= -0.03,0.254447


In [ ]:
#Her model için shap_importance.csv + perm_importance.csv dosyalarını okur

#Permutation dosyasında “importance” kolonunun adı neyse otomatik bulur (importance, mean_importance, perm_importance, vs.)

#SHAP’ta mean_abs_shap kolonunu kullanır

#Rank çıkarır, Spearman korelasyon hesaplar

#outputs/_shared/shap_vs_permutation_correlation.csv üretir

#Ek olarak “en çok uzlaşan feature’lar” için küçük özet CSV üretir

In [21]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr

MODELS = ["LogReg", "DecisionTree", "RandomForest", "XGBoost", "LightGBM", "CatBoost"]  # ✅ KNN yok
BASE = Path("outputs")  # sende BASE_OUT varsa onu kullanabilirsin: BASE_OUT

shared = BASE / "_shared"
shared.mkdir(parents=True, exist_ok=True)

def find_perm_col(df):
    """Permutation importance kolonunu otomatik bulur."""
    candidates = [
        "importance", "perm_importance", "mean_importance", "importances_mean",
        "permutation_importance", "mean", "score"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    # numeric kolonlar arasından en mantıklısını seç
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        raise ValueError(f"Permutation dosyasında numeric kolon yok. Kolonlar: {list(df.columns)}")
    # "feature" dışındaki ilk numeric kolon
    for c in num_cols:
        if c != "feature":
            return c
    return num_cols[0]

results = []
pairwise_top = []  # en çok uzlaşan feature’lar için

for m in MODELS:
    shap_path = BASE / m / "global" / "shap_importance.csv"
    perm_path = BASE / m / "global" / "perm_importance.csv"

    if not shap_path.exists():
        print(f"⚠️ SHAP yok: {m} -> {shap_path}")
        continue
    if not perm_path.exists():
        print(f"⚠️ Permutation yok: {m} -> {perm_path}")
        continue

    shap_df = pd.read_csv(shap_path)
    perm_df = pd.read_csv(perm_path)

    # SHAP kolon kontrol
    if "mean_abs_shap" not in shap_df.columns:
        raise ValueError(f"{m} shap_importance.csv içinde mean_abs_shap yok. Kolonlar: {list(shap_df.columns)}")

    # Permutation kolonunu bul
    perm_col = find_perm_col(perm_df)

    # Rank hesapla (1 en önemli)
    shap_df = shap_df[["feature", "mean_abs_shap"]].copy()
    perm_df = perm_df[["feature", perm_col]].copy().rename(columns={perm_col: "perm_importance"})

    shap_df["shap_rank"] = shap_df["mean_abs_shap"].rank(ascending=False, method="average")
    perm_df["perm_rank"] = perm_df["perm_importance"].rank(ascending=False, method="average")

    merged = shap_df.merge(perm_df, on="feature", how="inner")

    rho, pval = spearmanr(merged["shap_rank"], merged["perm_rank"])

    # Top-K ortaklık analizi (tez için güzel)
    K = 20
    top_shap = set(shap_df.sort_values("mean_abs_shap", ascending=False).head(K)["feature"])
    top_perm = set(perm_df.sort_values("perm_importance", ascending=False).head(K)["feature"])
    overlap = len(top_shap & top_perm)

    results.append({
        "model": m,
        "n_features_compared": int(merged.shape[0]),
        "spearman_rho": float(rho),
        "p_value": float(pval),
        "top20_overlap_count": int(overlap),
        "perm_col_used": perm_col
    })

    # en çok uzlaşan feature’lar: rank farkı küçük olanlar
    merged["rank_gap"] = (merged["shap_rank"] - merged["perm_rank"]).abs()
    top_agree = merged.sort_values("rank_gap").head(15).copy()
    top_agree["model"] = m
    pairwise_top.append(top_agree[["model","feature","shap_rank","perm_rank","rank_gap"]])

df_res = pd.DataFrame(results).sort_values("spearman_rho", ascending=False)
out_corr = shared / "shap_vs_permutation_correlation.csv"
df_res.to_csv(out_corr, index=False, encoding="utf-8")

print("✅ Korelasyon tablosu kaydedildi:", out_corr)
display(df_res)

if pairwise_top:
    df_agree = pd.concat(pairwise_top, ignore_index=True)
    out_agree = shared / "shap_vs_permutation_top_agreement.csv"
    df_agree.to_csv(out_agree, index=False, encoding="utf-8")
    print("✅ En çok uzlaşan feature’lar kaydedildi:", out_agree)

✅ Korelasyon tablosu kaydedildi: outputs\_shared\shap_vs_permutation_correlation.csv


,model,n_features_compared,spearman_rho,p_value,top20_overlap_count,perm_col_used
1,DecisionTree,75,0.967819,1.739524e-45,18,importance_mean
4,LightGBM,75,0.965563,1.982013e-44,17,importance_mean
5,CatBoost,75,0.943103,1.214374e-36,17,importance_mean
3,XGBoost,75,0.940904,4.662491e-36,17,importance_mean
0,LogReg,75,0.926225,1.179161e-32,19,importance_mean
2,RandomForest,75,0.861765,3.282675e-23,10,importance_mean


✅ En çok uzlaşan feature’lar kaydedildi: outputs\_shared\shap_vs_permutation_top_agreement.csv
